# 🔄 McMiner-in-the-Loop TRAVER Pipeline
## Round-by-Round Misconception Detection & Injection

After each student response, we:
1. Generate diagnostic code (Llama-3.1-8B, N=1)
2. Run McMiner (Gemini 2.5 Flash) to detect misconceptions
3. Inject misconceptions into the tutor prompt for the next round

**Projects:** easyvolcap (12), searcharray (6), xinhua (1), sd-forge (3)


---
## 0. GPU Check & Configuration

In [ ]:
# Check GPU (warns if no GPU attached - change runtime to T4 if needed)
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
else:
    print("⚠️  No GPU detected. Go to Runtime → Change runtime type → T4 GPU")

# Mount Google Drive (safe to re-run: skips if already mounted)
import os
from google.colab import drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
    print("✅ Google Drive mounted.")
else:
    print("✅ Google Drive already mounted.")

In [ ]:
# ===== LOAD API KEYS FROM COLAB SECRETS =====
# Go to the 🔑 icon in the left sidebar → add HF_TOKEN and GOOGLE_API_KEY
from google.colab import userdata
import os

HF_TOKEN = userdata.get('HF_TOKEN')
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')

# ===== MODEL CONFIGURATION =====
TUTOR_MODEL_ID = "meta-llama/Llama-3.1-70B-Instruct"
# Loaded locally on Colab GPU in 4-bit (pre-2024 cutoff, no data contamination)
STUDENT_MODEL_ID = f"{MODEL_DIR}/Mistral-7B-Instruct-v0.2"
# HF Inference Providers: single base URL, model goes in request body
HF_API_BASE = "https://router.huggingface.co/v1"

# ===== STORAGE =====
DRIVE_DIR = "/content/drive/MyDrive/Coding-Tutor-Colab"
WORK_DIR = "/content/Coding-Tutor"
MODEL_DIR = f"{DRIVE_DIR}/models"
DATA_DIR = f"{DRIVE_DIR}/data"

# ===== PIPELINE SETTINGS =====
STUDENT_LEVELS = ["low_level", "med_level", "high_level"]
TUTOR_NUM_RESPONSES = 5  # Best-of-5 with cache busting

# ===== DATASET IDS =====
TUTOR_AGENTS_DATASET = "nlpscu/Tutor-Agents"
MCMINER_DATASET = "nlpscu/MCminer"

# Validate
assert HF_TOKEN, "Add HF_TOKEN to Colab Secrets (🔑 sidebar)!"
assert GOOGLE_API_KEY, "Add GOOGLE_API_KEY to Colab Secrets (🔑 sidebar)!"
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
print("✅ Configuration loaded from Colab Secrets.")

---
## 1. Install Dependencies & Clone Repos

In [ ]:
# Clone Coding-Tutor repo
if not os.path.exists("/content/Coding-Tutor"):
    !git clone https://github.com/iwangjian/Coding-Tutor.git /content/Coding-Tutor
else:
    print("✅ Coding-Tutor already exists.")

# Clone McMiner repo
if not os.path.exists("/content/mcminer"):
    !git clone https://github.com/taisazero/mcminer.git /content/mcminer
else:
    print("✅ mcminer already exists.")

In [ ]:
# Install dependencies
# NOTE: torch is pre-installed on Colab — do NOT reinstall it or pin a version.
# Pinning transformers==4.44.2 also hardcodes torch==2.4.0 which conflicts
# with Colab's torch 2.10.0+cu128. We let pip resolve versions freely.

# Batch 1: Core ML libraries (compatible with Colab's existing torch)
!pip install -q bitsandbytes peft accelerate safetensors

# Batch 2: Transformers + tokenizers (no torch version pinned)
!pip install -q "transformers>=4.44,<4.48" tiktoken sentencepiece protobuf

# Batch 3: API clients
!pip install -q openai huggingface_hub tenacity google-generativeai

# Batch 4: Utilities
!pip install -q tqdm python-dotenv

# Verify torch version is still the Colab-native one
import torch
print(f"✅ All dependencies installed.")
print(f"✅ torch version: {torch.__version__} (should be 2.10.x)")

---
## 2. Download Datasets & Pre-trained Models
Saved to Google Drive — no re-downloading on session restart.

In [ ]:
from huggingface_hub import snapshot_download, login
login(token=HF_TOKEN)

# --- Download Tutor-Agents dataset ---
tutor_data_dir = f"{DATA_DIR}/Tutor-Agents"
if not os.path.exists(tutor_data_dir):
    print("⬇️  Downloading Tutor-Agents dataset...")
    snapshot_download(
        TUTOR_AGENTS_DATASET, repo_type="dataset",
        local_dir=tutor_data_dir, token=HF_TOKEN)
    print("✅ Tutor-Agents dataset downloaded.")
else:
    print("✅ Tutor-Agents dataset already on Drive.")

# --- Download MCminer dataset ---
mcminer_data_dir = f"{DATA_DIR}/MCminer"
if not os.path.exists(mcminer_data_dir):
    print("⬇️  Downloading MCminer dataset...")
    snapshot_download(
        MCMINER_DATASET, repo_type="dataset",
        local_dir=mcminer_data_dir, token=HF_TOKEN)
    print("✅ MCminer dataset downloaded.")
else:
    print("✅ MCminer dataset already on Drive.")

print(f"\n📦 Dataset storage:")
!du -sh {DATA_DIR}/*
# --- Download Student Model locally to Drive ---
student_dir = f"{MODEL_DIR}/Mistral-7B-Instruct-v0.2"
if not os.path.exists(student_dir) or len(os.listdir(student_dir)) < 5:
    print(f"Downloading Student Model to {student_dir}...")
    snapshot_download(repo_id="mistralai/Mistral-7B-Instruct-v0.2", local_dir=student_dir, ignore_patterns=["*.pth", "*.h5"])
else:
    print("Student Model already exists in Drive.")


In [ ]:
# --- Download pre-trained Verifier-7B (5 shards, ~1.5 GB) ---
verifier_dir = f"{MODEL_DIR}/Verifier-7B"
if not os.path.exists(f"{verifier_dir}/part0"):
    print("⬇️  Downloading Verifier-7B checkpoints...")
    snapshot_download("jwanglvy/Verifier-7B", local_dir=verifier_dir, token=HF_TOKEN)
    print("✅ Verifier-7B downloaded.")
else:
    print("✅ Verifier-7B already on Drive.")

# --- Download Mistral-7B base model (verifier architecture) ---
mistral_dir = f"{MODEL_DIR}/Mistral-7B-v0.1"
if not os.path.exists(f"{mistral_dir}/config.json"):
    print("⬇️  Downloading Mistral-7B-v0.1...")
    snapshot_download("mistralai/Mistral-7B-v0.1", local_dir=mistral_dir, token=HF_TOKEN)
    print("✅ Mistral-7B-v0.1 downloaded.")
else:
    print("✅ Mistral-7B-v0.1 already on Drive.")

print(f"\n📦 Models:")
!du -sh {MODEL_DIR}/*
# --- Download Student Model locally to Drive ---
student_dir = f"{MODEL_DIR}/Mistral-7B-Instruct-v0.2"
if not os.path.exists(student_dir) or len(os.listdir(student_dir)) < 5:
    print(f"Downloading Student Model to {student_dir}...")
    snapshot_download(repo_id="mistralai/Mistral-7B-Instruct-v0.2", local_dir=student_dir, ignore_patterns=["*.pth", "*.h5"])
else:
    print("Student Model already exists in Drive.")


---
## 3. Set Up Working Directory

In [ ]:
import shutil

# Symlink output directory to Drive for persistence
output_link = f"{WORK_DIR}/output"
drive_output = f"{DRIVE_DIR}/output"
os.makedirs(drive_output, exist_ok=True)

if os.path.islink(output_link):
    os.unlink(output_link)
elif os.path.isdir(output_link):
    !cp -rn {output_link}/* {drive_output}/ 2>/dev/null; rm -rf {output_link}
os.symlink(drive_output, output_link)

# Copy MCminer data files into mcminer repo
for fname in ["misconception_bank.json", "problems_processed.json"]:
    src = f"{DATA_DIR}/MCminer/{fname}"
    dst = f"/content/mcminer/{fname}"
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy2(src, dst)

# Copy corrupted_codes_best
src_dir = f"{DATA_DIR}/MCminer/corrupted_codes_best"
dst_dir = f"/content/mcminer/corrupted_codes_best"
if os.path.exists(src_dir) and not os.path.exists(dst_dir):
    shutil.copytree(src_dir, dst_dir)

print("✅ Working directories ready.")
print(f"  Output → {drive_output}")

---
## 4. Patch VLLMChat for HuggingFace Inference API

The original code expects local vLLM servers. We patch `VLLMChat` to:
- Use HF Inference API's OpenAI-compatible endpoint
- Handle `n > 1` by looping (HF doesn't support multi-completion)
- Add retry logic with exponential backoff

In [ ]:
import sys, base64, os
sys.path.insert(0, WORK_DIR)
sys.path.insert(0, f"{WORK_DIR}/traver")

# ── 1. Patch VLLMChat: HF API for tutor, LOCAL 4-bit for student ──
PATCH_FILE = f"{WORK_DIR}/traver/chatarena/backends/openai_vllm.py"
_patch_b64 = "IyBDT0xBQl9QQVRDSEVEIC0gSEYgQVBJIGZvciB0dXRvciArIExPQ0FMIDQtYml0IGZvciBzdHVkZW50CmZyb20gdHlwaW5nIGltcG9ydCBMaXN0CmltcG9ydCBvcywgcmUsIHRpbWUKZnJvbSBvcGVuYWkgaW1wb3J0IE9wZW5BSQpmcm9tIC5iYXNlIGltcG9ydCBJbnRlbGxpZ2VuY2VCYWNrZW5kCmZyb20gLi5tZXNzYWdlIGltcG9ydCBNZXNzYWdlLCBTWVNURU1fTkFNRQoKRU5EX09GX01FU1NBR0UgPSAiPEVPUz4iCgpfTE9DQUxfTU9ERUxfQ0FDSEUgPSB7fQoKZGVmIF9sb2FkX2xvY2FsX21vZGVsKG1vZGVsX2lkLCBoZl90b2tlbik6CiAgICBpZiBtb2RlbF9pZCBpbiBfTE9DQUxfTU9ERUxfQ0FDSEU6CiAgICAgICAgcHJpbnQoZiIgIFtWTExNQ2hhdF0gUmV1c2luZyBjYWNoZWQgbG9jYWwgbW9kZWw6IHttb2RlbF9pZH0iKQogICAgICAgIHJldHVybiBfTE9DQUxfTU9ERUxfQ0FDSEVbbW9kZWxfaWRdCiAgICBpbXBvcnQgdG9yY2gKICAgIGZyb20gdHJhbnNmb3JtZXJzIGltcG9ydCBBdXRvTW9kZWxGb3JDYXVzYWxMTSwgQXV0b1Rva2VuaXplciwgQml0c0FuZEJ5dGVzQ29uZmlnCiAgICBwcmludChmIiAgW1ZMTE1DaGF0XSBMb2FkaW5nIGxvY2FsIG1vZGVsIGluIDQtYml0OiB7bW9kZWxfaWR9IikKICAgIGJuYl9jb25maWcgPSBCaXRzQW5kQnl0ZXNDb25maWcoCiAgICAgICAgbG9hZF9pbl80Yml0PVRydWUsCiAgICAgICAgYm5iXzRiaXRfcXVhbnRfdHlwZT0ibmY0IiwKICAgICAgICBibmJfNGJpdF9jb21wdXRlX2R0eXBlPXRvcmNoLmZsb2F0MTYsCiAgICApCiAgICB0b2tlbml6ZXIgPSBBdXRvVG9rZW5pemVyLmZyb21fcHJldHJhaW5lZChtb2RlbF9pZCwgdG9rZW49aGZfdG9rZW4sIHRydXN0X3JlbW90ZV9jb2RlPVRydWUpCiAgICBpZiB0b2tlbml6ZXIucGFkX3Rva2VuIGlzIE5vbmU6CiAgICAgICAgdG9rZW5pemVyLnBhZF90b2tlbiA9IHRva2VuaXplci5lb3NfdG9rZW4KICAgIG1vZGVsID0gQXV0b01vZGVsRm9yQ2F1c2FsTE0uZnJvbV9wcmV0cmFpbmVkKAogICAgICAgIG1vZGVsX2lkLAogICAgICAgIHF1YW50aXphdGlvbl9jb25maWc9Ym5iX2NvbmZpZywKICAgICAgICBkZXZpY2VfbWFwPSJhdXRvIiwKICAgICAgICB0b3JjaF9kdHlwZT10b3JjaC5mbG9hdDE2LAogICAgICAgIHRva2VuPWhmX3Rva2VuLAogICAgICAgIHRydXN0X3JlbW90ZV9jb2RlPVRydWUsCiAgICAgICAgbG93X2NwdV9tZW1fdXNhZ2U9VHJ1ZSwKICAgICkKICAgIG1vZGVsLmV2YWwoKQogICAgcHJpbnQoZiIgIFtWTExNQ2hhdF0gTG9jYWwgbW9kZWwgcmVhZHkiKQogICAgX0xPQ0FMX01PREVMX0NBQ0hFW21vZGVsX2lkXSA9IChtb2RlbCwgdG9rZW5pemVyKQogICAgcmV0dXJuIG1vZGVsLCB0b2tlbml6ZXIKCmNsYXNzIFZMTE1DaGF0KEludGVsbGlnZW5jZUJhY2tlbmQpOgogICAgc3RhdGVmdWwgPSBGYWxzZQogICAgdHlwZV9uYW1lID0gInZsbG0tY2hhdCIKCiAgICBkZWYgX19pbml0X18oc2VsZiwgdmxsbV9hcGlfa2V5LCB2bGxtX2VuZHBvaW50LCBtb2RlbF9uYW1lX29yX3BhdGgsCiAgICAgICAgICAgICAgICAgdGVtcGVyYXR1cmU9MC43NSwgdG9wX3A9MC45NSwgbWF4X3Rva2Vucz01MDAsCiAgICAgICAgICAgICAgICAgbWF4X2xhdGVzdF9tZXNzYWdlcz0tMSwgKiprd2FyZ3MpOgogICAgICAgIHN1cGVyKCkuX19pbml0X18obW9kZWxfbmFtZV9vcl9wYXRoPW1vZGVsX25hbWVfb3JfcGF0aCwKICAgICAgICAgICAgICAgICAgICAgICAgIHRlbXBlcmF0dXJlPXRlbXBlcmF0dXJlLCB0b3BfcD10b3BfcCwKICAgICAgICAgICAgICAgICAgICAgICAgIG1heF90b2tlbnM9bWF4X3Rva2VucywKICAgICAgICAgICAgICAgICAgICAgICAgIG1heF9sYXRlc3RfbWVzc2FnZXM9bWF4X2xhdGVzdF9tZXNzYWdlcywgKiprd2FyZ3MpCiAgICAgICAgc2VsZi5tb2RlbCA9IG1vZGVsX25hbWVfb3JfcGF0aAogICAgICAgIHNlbGYudGVtcGVyYXR1cmUgPSB0ZW1wZXJhdHVyZQogICAgICAgIHNlbGYudG9wX3AgPSB0b3BfcAogICAgICAgIHNlbGYubWF4X3Rva2VucyA9IG1heF90b2tlbnMKICAgICAgICBzZWxmLm1heF9sYXRlc3RfbWVzc2FnZXMgPSBtYXhfbGF0ZXN0X21lc3NhZ2VzCiAgICAgICAgc2VsZi5faXNfbG9jYWwgPSAodmxsbV9lbmRwb2ludCA9PSAibG9jYWwiKQogICAgICAgIGlmIHNlbGYuX2lzX2xvY2FsOgogICAgICAgICAgICBzZWxmLmxvY2FsX21vZGVsLCBzZWxmLmxvY2FsX3Rva2VuaXplciA9IF9sb2FkX2xvY2FsX21vZGVsKG1vZGVsX25hbWVfb3JfcGF0aCwgdmxsbV9hcGlfa2V5KQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHNlbGYuY2xpZW50ID0gT3BlbkFJKGFwaV9rZXk9dmxsbV9hcGlfa2V5LCBiYXNlX3VybD12bGxtX2VuZHBvaW50KQoKICAgIGRlZiBfbG9jYWxfY2FsbChzZWxmLCBtZXNzYWdlcyk6CiAgICAgICAgaW1wb3J0IHRvcmNoCiAgICAgICAgdHJ5OgogICAgICAgICAgICBwcm9tcHQgPSBzZWxmLmxvY2FsX3Rva2VuaXplci5hcHBseV9jaGF0X3RlbXBsYXRlKAogICAgICAgICAgICAgICAgbWVzc2FnZXMsIHRva2VuaXplPUZhbHNlLCBhZGRfZ2VuZXJhdGlvbl9wcm9tcHQ9VHJ1ZSkKICAgICAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgICAgICBwYXJ0cyA9IFtdCiAgICAgICAgICAgIGZvciBtIGluIG1lc3NhZ2VzOgogICAgICAgICAgICAgICAgcGFydHMuYXBwZW5kKG0uZ2V0KCJyb2xlIiwgInVzZXIiKSArICI6ICIgKyBtLmdldCgiY29udGVudCIsICIiKSkKICAgICAgICAgICAgcHJvbXB0ID0gIlxuIi5qb2luKHBhcnRzKSArICJcbmFzc2lzdGFudDogIgogICAgICAgIGlucHV0cyA9IHNlbGYubG9jYWxfdG9rZW5pemVyKHByb21wdCwgcmV0dXJuX3RlbnNvcnM9InB0IiwgdHJ1bmNhdGlvbj1UcnVlLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBtYXhfbGVuZ3RoPTIwNDgpLnRvKHNlbGYubG9jYWxfbW9kZWwuZGV2aWNlKQogICAgICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOgogICAgICAgICAgICBvdXRwdXRzID0gc2VsZi5sb2NhbF9tb2RlbC5nZW5lcmF0ZSgKICAgICAgICAgICAgICAgICoqaW5wdXRzLAogICAgICAgICAgICAgICAgbWF4X25ld190b2tlbnM9c2VsZi5tYXhfdG9rZW5zLAogICAgICAgICAgICAgICAgdGVtcGVyYXR1cmU9bWF4KHNlbGYudGVtcGVyYXR1cmUsIDAuMDEpLAogICAgICAgICAgICAgICAgdG9wX3A9c2VsZi50b3BfcCwKICAgICAgICAgICAgICAgIGRvX3NhbXBsZT1UcnVlLAogICAgICAgICAgICAgICAgcGFkX3Rva2VuX2lkPXNlbGYubG9jYWxfdG9rZW5pemVyLmVvc190b2tlbl9pZCwKICAgICAgICAgICAgKQogICAgICAgIG5ld190b2tlbnMgPSBvdXRwdXRzWzBdW2lucHV0c1siaW5wdXRfaWRzIl0uc2hhcGVbMV06XQogICAgICAgIHJlc3BvbnNlID0gc2VsZi5sb2NhbF90b2tlbml6ZXIuZGVjb2RlKG5ld190b2tlbnMsIHNraXBfc3BlY2lhbF90b2tlbnM9VHJ1ZSkKICAgICAgICByZXR1cm4gcmVzcG9uc2Uuc3RyaXAoKQoKICAgIGRlZiBfYXBpX2NhbGwoc2VsZiwgbWVzc2FnZXMpOgogICAgICAgIGZvciBhdHRlbXB0IGluIHJhbmdlKDUpOgogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICBjID0gc2VsZi5jbGllbnQuY2hhdC5jb21wbGV0aW9ucy5jcmVhdGUoCiAgICAgICAgICAgICAgICAgICAgbW9kZWw9c2VsZi5tb2RlbCwgbWVzc2FnZXM9bWVzc2FnZXMsCiAgICAgICAgICAgICAgICAgICAgdGVtcGVyYXR1cmU9c2VsZi50ZW1wZXJhdHVyZSwgdG9wX3A9c2VsZi50b3BfcCwKICAgICAgICAgICAgICAgICAgICBtYXhfdG9rZW5zPXNlbGYubWF4X3Rva2Vucywgbj0xKQogICAgICAgICAgICAgICAgciA9IGMuY2hvaWNlc1swXS5tZXNzYWdlLmNvbnRlbnQKICAgICAgICAgICAgICAgIHJldHVybiByLnN0cmlwKCkgaWYgciBlbHNlICIiCiAgICAgICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToKICAgICAgICAgICAgICAgIHdhaXQgPSA1ICogKGF0dGVtcHQgKyAxKQogICAgICAgICAgICAgICAgcHJpbnQoIiAgQVBJIGVycm9yIChhdHRlbXB0ICIgKyBzdHIoYXR0ZW1wdCArIDEpICsgIi81KTogIiArIHN0cihlKVs6MzAwXSArICIuIFdhaXRpbmcgIiArIHN0cih3YWl0KSArICJzLi4uIikKICAgICAgICAgICAgICAgIHRpbWUuc2xlZXAod2FpdCkKICAgICAgICByZXR1cm4gIltBUEkgRXJyb3JdIgoKICAgIGRlZiBfc2luZ2xlX2NhbGwoc2VsZiwgbWVzc2FnZXMpOgogICAgICAgIGlmIHNlbGYuX2lzX2xvY2FsOgogICAgICAgICAgICByZXR1cm4gc2VsZi5fbG9jYWxfY2FsbChtZXNzYWdlcykKICAgICAgICByZXR1cm4gc2VsZi5fYXBpX2NhbGwobWVzc2FnZXMpCgogICAgZGVmIF9nZXRfcmVzcG9uc2Uoc2VsZiwgbWVzc2FnZXMsIG51bV9yZXNwb25zZXM9MSk6CiAgICAgICAgaW1wb3J0IGNvcHksIHV1aWQKICAgICAgICBpZiBudW1fcmVzcG9uc2VzID4gMToKICAgICAgICAgICAgcmVzcG9uc2VzID0gW10KICAgICAgICAgICAgZm9yIGkgaW4gcmFuZ2UobnVtX3Jlc3BvbnNlcyk6CiAgICAgICAgICAgICAgICAjIEluamVjdCB1bmlxdWUgc2VlZCB0byBidXN0IEh1Z2dpbmdGYWNlIHJlc3BvbnNlIGNhY2hlCiAgICAgICAgICAgICAgICBtc2dzID0gY29weS5kZWVwY29weShtZXNzYWdlcykKICAgICAgICAgICAgICAgIHNlZWQgPSAiXG5bc2VlZDoiICsgdXVpZC51dWlkNCgpLmhleFs6OF0gKyAiXSIKICAgICAgICAgICAgICAgIG1zZ3NbLTFdWyJjb250ZW50Il0gPSBtc2dzWy0xXVsiY29udGVudCJdICsgc2VlZAogICAgICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgICAgIHJlc3BvbnNlcy5hcHBlbmQoc2VsZi5fc2luZ2xlX2NhbGwobXNncykpCiAgICAgICAgICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6CiAgICAgICAgICAgICAgICAgICAgcHJpbnQoZiIgIFtWTExNQ2hhdF0gQ2FuZGlkYXRlIHtpKzF9L3tudW1fcmVzcG9uc2VzfSBmYWlsZWQ6IHtlfSIpCiAgICAgICAgICAgICAgICAgICAgcmVzcG9uc2VzLmFwcGVuZCgiW0FQSSBFcnJvcl0iKQogICAgICAgICAgICAgICAgaWYgaSA8IG51bV9yZXNwb25zZXMgLSAxOgogICAgICAgICAgICAgICAgICAgIHRpbWUuc2xlZXAoMC4zKQogICAgICAgICAgICByZXR1cm4gcmVzcG9uc2VzCiAgICAgICAgcmV0dXJuIHNlbGYuX3NpbmdsZV9jYWxsKG1lc3NhZ2VzKQoKICAgIGRlZiBxdWVyeShzZWxmLCBhZ2VudF9uYW1lLCByb2xlX2Rlc2MsIGhpc3RvcnlfbWVzc2FnZXMsIGdsb2JhbF9wcm9tcHQ9Tm9uZSwKICAgICAgICAgICAgICByZXF1ZXN0X21zZz1Ob25lLCBudW1fcmVzcG9uc2VzPTEsICphcmdzLCAqKmt3YXJncyk6CiAgICAgICAgaWYgZ2xvYmFsX3Byb21wdDoKICAgICAgICAgICAgc3lzdGVtX3Byb21wdCA9IGdsb2JhbF9wcm9tcHQuc3RyaXAoKSArICJcblxuWW91ciBuYW1lOiAiICsgYWdlbnRfbmFtZSArICJcblxuWW91ciByb2xlOiAiICsgcm9sZV9kZXNjCiAgICAgICAgZWxzZToKICAgICAgICAgICAgc3lzdGVtX3Byb21wdCA9IHJvbGVfZGVzYwogICAgICAgIGlmIHNlbGYubWF4X2xhdGVzdF9tZXNzYWdlcyA+IDAgYW5kIGxlbihoaXN0b3J5X21lc3NhZ2VzKSA+IHNlbGYubWF4X2xhdGVzdF9tZXNzYWdlczoKICAgICAgICAgICAgaGlzdG9yeV9tZXNzYWdlcyA9IGhpc3RvcnlfbWVzc2FnZXNbLXNlbGYubWF4X2xhdGVzdF9tZXNzYWdlczpdCiAgICAgICAgYWxsX21lc3NhZ2VzID0gWyhTWVNURU1fTkFNRSwgc3lzdGVtX3Byb21wdCldCiAgICAgICAgZm9yIG1zZyBpbiBoaXN0b3J5X21lc3NhZ2VzOgogICAgICAgICAgICBpZiBtc2cuYWdlbnRfbmFtZSA9PSBTWVNURU1fTkFNRToKICAgICAgICAgICAgICAgIGFsbF9tZXNzYWdlcy5hcHBlbmQoKFNZU1RFTV9OQU1FLCBtc2cuY29udGVudCkpCiAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICBhbGxfbWVzc2FnZXMuYXBwZW5kKChtc2cuYWdlbnRfbmFtZSwgbXNnLmNvbnRlbnQgKyBFTkRfT0ZfTUVTU0FHRSkpCiAgICAgICAgaWYgcmVxdWVzdF9tc2cgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIGFsbF9tZXNzYWdlcy5hcHBlbmQoKFNZU1RFTV9OQU1FLCByZXF1ZXN0X21zZy5jb250ZW50KSkKICAgICAgICBlbHNlOgogICAgICAgICAgICBhbGxfbWVzc2FnZXMuYXBwZW5kKChTWVNURU1fTkFNRSwgIk5vdyB5b3Ugc3BlYWssICIgKyBhZ2VudF9uYW1lICsgIi4iICsgRU5EX09GX01FU1NBR0UpKQogICAgICAgIG1lc3NhZ2VzID0gW10KICAgICAgICBmb3IgaSwgbXNnIGluIGVudW1lcmF0ZShhbGxfbWVzc2FnZXMpOgogICAgICAgICAgICBpZiBpID09IDA6CiAgICAgICAgICAgICAgICBhc3NlcnQgbXNnWzBdID09IFNZU1RFTV9OQU1FCiAgICAgICAgICAgICAgICBtZXNzYWdlcy5hcHBlbmQoeyJyb2xlIjogInVzZXIiLCAiY29udGVudCI6IG1zZ1sxXX0pCiAgICAgICAgICAgIGVsaWYgaSA9PSBsZW4oYWxsX21lc3NhZ2VzKSAtIDE6CiAgICAgICAgICAgICAgICBhc3NlcnQgbXNnWzBdID09IFNZU1RFTV9OQU1FCiAgICAgICAgICAgICAgICBtZXNzYWdlc1stMV1bImNvbnRlbnQiXSA9IG1lc3NhZ2VzWy0xXVsiY29udGVudCJdICsgIlxuXG4iICsgbXNnWzFdCiAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICBpZiBtc2dbMF0gPT0gYWdlbnRfbmFtZToKICAgICAgICAgICAgICAgICAgICBtZXNzYWdlcy5hcHBlbmQoeyJyb2xlIjogImFzc2lzdGFudCIsICJjb250ZW50IjogbXNnWzFdfSkKICAgICAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICAgICAgaWYgbWVzc2FnZXNbLTFdWyJyb2xlIl0gPT0gInVzZXIiOgogICAgICAgICAgICAgICAgICAgICAgICBtZXNzYWdlc1stMV1bImNvbnRlbnQiXSA9IG1lc3NhZ2VzWy0xXVsiY29udGVudCJdICsgIlxuXG5bIiArIG1zZ1swXSArICJdOiAiICsgbXNnWzFdCiAgICAgICAgICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgICAgICAgICAgbWVzc2FnZXMuYXBwZW5kKHsicm9sZSI6ICJ1c2VyIiwgImNvbnRlbnQiOiAiWyIgKyBtc2dbMF0gKyAiXTogIiArIG1zZ1sxXX0pCiAgICAgICAgcmVzcG9uc2UgPSBzZWxmLl9nZXRfcmVzcG9uc2UobWVzc2FnZXMsIG51bV9yZXNwb25zZXMsICphcmdzLCAqKmt3YXJncykKICAgICAgICBpZiBudW1fcmVzcG9uc2VzID4gMToKICAgICAgICAgICAgcmVzcG9uc2UgPSBbcmUuc3ViKHIiXlxzKlxbLipdOiIsICIiLCByKS5zdHJpcCgpIGZvciByIGluIHJlc3BvbnNlXQogICAgICAgICAgICByZXNwb25zZSA9IFtyZS5zdWIociJeXHMqIiArIHJlLmVzY2FwZShhZ2VudF9uYW1lKSArIHIiXHMqOiIsICIiLCByKS5zdHJpcCgpIGZvciByIGluIHJlc3BvbnNlXQogICAgICAgICAgICByZXNwb25zZSA9IFtyZS5zdWIoRU5EX09GX01FU1NBR0UgKyAiJCIsICIiLCByKS5zdHJpcCgpIGZvciByIGluIHJlc3BvbnNlXQogICAgICAgICAgICBmb3IgaWR4LCByIGluIGVudW1lcmF0ZShyZXNwb25zZSk6CiAgICAgICAgICAgICAgICBpZiBFTkRfT0ZfTUVTU0FHRSBpbiByOgogICAgICAgICAgICAgICAgICAgIHJlc3BvbnNlW2lkeF0gPSByLnNwbGl0KEVORF9PRl9NRVNTQUdFKVswXS5zdHJpcCgpCiAgICAgICAgZWxzZToKICAgICAgICAgICAgcmVzcG9uc2UgPSByZS5zdWIociJeXHMqXFsuKl06IiwgIiIsIHJlc3BvbnNlKS5zdHJpcCgpCiAgICAgICAgICAgIHJlc3BvbnNlID0gcmUuc3ViKHIiXlxzKiIgKyByZS5lc2NhcGUoYWdlbnRfbmFtZSkgKyByIlxzKjoiLCAiIiwgcmVzcG9uc2UpLnN0cmlwKCkKICAgICAgICAgICAgcmVzcG9uc2UgPSByZS5zdWIoRU5EX09GX01FU1NBR0UgKyAiJCIsICIiLCByZXNwb25zZSkuc3RyaXAoKQogICAgICAgICAgICBpZiBFTkRfT0ZfTUVTU0FHRSBpbiByZXNwb25zZToKICAgICAgICAgICAgICAgIHJlc3BvbnNlID0gcmVzcG9uc2Uuc3BsaXQoRU5EX09GX01FU1NBR0UpWzBdLnN0cmlwKCkKICAgICAgICByZXR1cm4gcmVzcG9uc2UK"
patched = base64.b64decode(_patch_b64.encode('ascii')).decode('utf-8')
with open(PATCH_FILE, 'w') as f:
    f.write(patched)
print("✅ VLLMChat patched: Tutor→HF API, Student→LOCAL 4-bit GPU")

print("Applying OOM & 4-Bit Fixes for Phase 1...")

# ── Overwrite model_utils.py with the proven working version ──
# (Same fix as the working colab_traver_mcminer.ipynb)
_mu_b64 = "aW1wb3J0IG9zCmltcG9ydCB0b3JjaApmcm9tIHR5cGluZyBpbXBvcnQgTGlzdCwgT3B0aW9uYWwKZnJvbSB0cmFuc2Zvcm1lcnMgaW1wb3J0ICgKICAgIEF1dG9Nb2RlbEZvckNhdXNhbExNLCAKICAgIEF1dG9Ub2tlbml6ZXIsIAogICAgQml0c0FuZEJ5dGVzQ29uZmlnLAogICAgVHJhaW5lciwKKQpmcm9tIHBlZnQgaW1wb3J0ICgKICAgIExvcmFDb25maWcsCiAgICBwcmVwYXJlX21vZGVsX2Zvcl9rYml0X3RyYWluaW5nLAopCmZyb20gLm1vZGVsIGltcG9ydCBWZXJpZmllcgoKCmNsYXNzIFZlcmlmaWVyVHJhaW5lcihUcmFpbmVyKToKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBtb2RlbCwgYXJncywgdG9rZW5pemVyLCB0cmFpbl9kYXRhc2V0LCBldmFsX2RhdGFzZXQpOgogICAgICAgIHN1cGVyKCkuX19pbml0X18obW9kZWwsIGFyZ3MsCiAgICAgICAgICAgICAgICAgICAgICAgICB0b2tlbml6ZXI9dG9rZW5pemVyLAogICAgICAgICAgICAgICAgICAgICAgICAgdHJhaW5fZGF0YXNldD10cmFpbl9kYXRhc2V0LAogICAgICAgICAgICAgICAgICAgICAgICAgZXZhbF9kYXRhc2V0PWV2YWxfZGF0YXNldCkKCiAgICBkZWYgc2F2ZV9tb2RlbChzZWxmLCBvdXRwdXRfZGlyOiBPcHRpb25hbFtzdHJdID0gTm9uZSwgX2ludGVybmFsX2NhbGw6IGJvb2wgPSBGYWxzZSk6CiAgICAgICAgaWYgb3V0cHV0X2RpciBpcyBOb25lOgogICAgICAgICAgICBvdXRwdXRfZGlyID0gc2VsZi5hcmdzLm91dHB1dF9kaXIKICAgICAgICBvcy5tYWtlZGlycyhvdXRwdXRfZGlyLCBleGlzdF9vaz1UcnVlKQoKICAgICAgICBtb2RlbF90b19zYXZlID0gc2VsZi5tb2RlbAoKICAgICAgICBvdXRwdXRfbW9kZWxfZmlsZSA9IG9zLnBhdGguam9pbihvdXRwdXRfZGlyLCAicHl0b3JjaF9tb2RlbC5iaW4iKQogICAgICAgIHRvcmNoLnNhdmUobW9kZWxfdG9fc2F2ZS5zdGF0ZV9kaWN0KCksIG91dHB1dF9tb2RlbF9maWxlKQoKCmRlZiBsb2FkX21vZGVsKAogICAgYmFzZV9tb2RlbF9uYW1lX29yX3BhdGg6IHN0ciwKICAgIHRyYWluZWRfdmVyaWZpZXJfbW9kZWxfcGF0aDogc3RyID0gTm9uZSwKICAgIGxvcmFfcjogaW50ID0gOCwKICAgIGxvcmFfYWxwaGE6IGludCA9IDE2LAogICAgbG9yYV9kcm9wb3V0OiBmbG9hdCA9IDAuMDUsCiAgICBsb3JhX3RhcmdldF9tb2R1bGVzOiBMaXN0W3N0cl0gPSAgWyJxX3Byb2oiLCAidl9wcm9qIl0sCiAgICBmcDE2OiBib29sID0gVHJ1ZSwKICAgIGJmMTY6IGJvb2wgPSBGYWxzZSwKICAgIGdyYWRpZW50X2NoZWNrcG9pbnRpbmc6IGJvb2wgPSBGYWxzZQopOgogICAgIyBMb2FkIHRoZSBwcmUtdHJhaW5lZCBtb2RlbCBhbmQgdG9rZW5pemVyCiAgICBkZXZpY2VfbWFwID0geyIiOiAwfSAgIyBGb3JjZSBhbGwgb24gR1BVLCBubyBDUFUgb2ZmbG9hZAogICAgd29ybGRfc2l6ZSA9IGludChvcy5lbnZpcm9uLmdldCgiV09STERfU0laRSIsIDEpKQogICAgZGRwID0gd29ybGRfc2l6ZSAhPSAxCiAgICBpZiBkZHA6CiAgICAgICAgZGV2aWNlX21hcCA9IHsiIjogaW50KG9zLmVudmlyb24uZ2V0KCJMT0NBTF9SQU5LIikgb3IgMCl9CiAgICAKICAgIGNvbXB1dGVfZHR5cGUgPSAoCiAgICAgICAgdG9yY2guZmxvYXQxNgogICAgICAgIGlmIGZwMTYKICAgICAgICBlbHNlICh0b3JjaC5iZmxvYXQxNiBpZiBiZjE2IGVsc2UgdG9yY2guZmxvYXQzMikKICAgICkgICAgCgogICAgIyA0LWJpdCBxdWFudGl6YXRpb24gZm9yIENvbGFiIFQ0L1YxMDAgKDE1LTE2R0IgVlJBTSkKICAgIGJuYl9jb25maWcgPSBCaXRzQW5kQnl0ZXNDb25maWcoCiAgICAgICAgbG9hZF9pbl80Yml0PVRydWUsCiAgICAgICAgYm5iXzRiaXRfcXVhbnRfdHlwZT0ibmY0IiwKICAgICAgICBibmJfNGJpdF9jb21wdXRlX2R0eXBlPWNvbXB1dGVfZHR5cGUsCiAgICAgICAgYm5iXzRiaXRfdXNlX2RvdWJsZV9xdWFudD1GYWxzZSwKICAgICkKCiAgICBiYXNlX21vZGVsID0gQXV0b01vZGVsRm9yQ2F1c2FsTE0uZnJvbV9wcmV0cmFpbmVkKAogICAgICAgIGJhc2VfbW9kZWxfbmFtZV9vcl9wYXRoLAogICAgICAgIHF1YW50aXphdGlvbl9jb25maWc9Ym5iX2NvbmZpZywKICAgICAgICBkZXZpY2VfbWFwPWRldmljZV9tYXAsCiAgICAgICAgdG9yY2hfZHR5cGU9Y29tcHV0ZV9kdHlwZSwKICAgICAgICB1c2VfY2FjaGU9RmFsc2UsCiAgICAgICAgdHJ1c3RfcmVtb3RlX2NvZGU9VHJ1ZQogICAgKQogICAgbG9yYV9jb25maWcgPSBMb3JhQ29uZmlnKAogICAgICAgIHI9bG9yYV9yLAogICAgICAgIGxvcmFfYWxwaGE9bG9yYV9hbHBoYSwKICAgICAgICB0YXJnZXRfbW9kdWxlcz1sb3JhX3RhcmdldF9tb2R1bGVzLAogICAgICAgIGxvcmFfZHJvcG91dD1sb3JhX2Ryb3BvdXQsCiAgICAgICAgYmlhcz0ibm9uZSIsCiAgICAgICAgdGFza190eXBlPSJDQVVTQUxfTE0iLAogICAgKQoKICAgICMgT25seSBwcmVwYXJlIGZvciB0cmFpbmluZyAoY2FzdHMgdG8gZnAzMiBmb3Igc3RhYmxlIGdyYWRpZW50cykuCiAgICAjIFNraXAgZHVyaW5nIGluZmVyZW5jZSB0byBzYXZlIFZSQU0g4oCUIG5vIGJhY2twcm9wIG5lZWRlZC4KICAgIGlmIHRyYWluZWRfdmVyaWZpZXJfbW9kZWxfcGF0aCBpcyBOb25lOgogICAgICAgIGJhc2VfbW9kZWwgPSBwcmVwYXJlX21vZGVsX2Zvcl9rYml0X3RyYWluaW5nKAogICAgICAgICAgICBiYXNlX21vZGVsLCB1c2VfZ3JhZGllbnRfY2hlY2twb2ludGluZz1ncmFkaWVudF9jaGVja3BvaW50aW5nKQogICAgCiAgICBpZiBub3QgZGRwIGFuZCB0b3JjaC5jdWRhLmRldmljZV9jb3VudCgpID4gMToKICAgICAgICAjIGtlZXBzIFRyYWluZXIgZnJvbSB0cnlpbmcgaXRzIG93biBEYXRhUGFyYWxsZWxpc20gd2hlbiBtb3JlIHRoYW4gMSBncHUgaXMgYXZhaWxhYmxlCiAgICAgICAgYmFzZV9tb2RlbC5pc19wYXJhbGxlbGl6YWJsZSA9IFRydWUKICAgICAgICBiYXNlX21vZGVsLm1vZGVsX3BhcmFsbGVsID0gVHJ1ZQoKICAgICMgU2V0IHRva2VuaXplcidzIHBhZGRpbmcgdG9rZW4gYW5kIHBhZGRpbmcgc2lkZQogICAgdG9rZW5pemVyID0gQXV0b1Rva2VuaXplci5mcm9tX3ByZXRyYWluZWQoCiAgICAgICAgYmFzZV9tb2RlbF9uYW1lX29yX3BhdGgsCiAgICAgICAgdHJ1bmNhdGlvbl9zaWRlPSdsZWZ0JywgICMgc2V0IHRvICdsZWZ0JyB0byB0cnVuY2F0ZSB0aGUgaW5wdXQgZnJvbSB0aGUgbGVmdAogICAgICAgIHRydXN0X3JlbW90ZV9jb2RlPVRydWUKICAgICkKICAgIGlmIGJhc2VfbW9kZWwuY29uZmlnLm1vZGVsX3R5cGUgPT0gImxsYW1hIiBvciBiYXNlX21vZGVsLmNvbmZpZy5tb2RlbF90eXBlID09ICJtaXN0cmFsIjoKICAgICAgICB0b2tlbml6ZXIucGFkX3Rva2VuID0gdG9rZW5pemVyLmVvc190b2tlbgoKICAgICMgV3JhcCB0aGUgbW9kZWwgd2l0aCB0aGUgZGVmaW5lZCBQUk0gbW9kZWwKICAgIHZlcmlmeV9tb2RlbCA9IFZlcmlmaWVyKAogICAgICAgIG1vZGVsPWJhc2VfbW9kZWwsCiAgICAgICAgbG9yYV9jb25maWc9bG9yYV9jb25maWcsCiAgICAgICAgdG9yY2hfZHR5cGU9Y29tcHV0ZV9kdHlwZQogICAgKQoKICAgIGlmIHRyYWluZWRfdmVyaWZpZXJfbW9kZWxfcGF0aCBpcyBub3QgTm9uZToKICAgICAgICBwcmludChmIkxvYWRpbmcgdHJhaW5lZCB2ZXJpZmllciBtb2RlbCBmcm9tIHt0cmFpbmVkX3ZlcmlmaWVyX21vZGVsX3BhdGh9IikKICAgICAgICBzdGF0ZV9kaWN0ID0gdG9yY2gubG9hZCh0cmFpbmVkX3ZlcmlmaWVyX21vZGVsX3BhdGgsIG1hcF9sb2NhdGlvbj0iY3B1Iiwgd2VpZ2h0c19vbmx5PVRydWUpCiAgICAgICAgdHJ5OgogICAgICAgICAgICB2ZXJpZnlfbW9kZWwubG9hZF9zdGF0ZV9kaWN0KHN0YXRlX2RpY3QsIHN0cmljdD1GYWxzZSkKICAgICAgICAgICAgcHJpbnQoIk1vZGVsIGxvYWRlZCBzdWNjZXNzZnVsbHkiKQogICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToKICAgICAgICAgICAgcHJpbnQoZiJXYXJuaW5nOiBFcnJvciBsb2FkaW5nIG1vZGVsIHN0YXRlIGRpY3Q6IHtlfSIpCiAgICAgICAgICAgIG1pc3Npbmdfa2V5cywgdW5leHBlY3RlZF9rZXlzID0gdmVyaWZ5X21vZGVsLmxvYWRfc3RhdGVfZGljdChzdGF0ZV9kaWN0LCBzdHJpY3Q9RmFsc2UpCiAgICAgICAgICAgIGlmIG1pc3Npbmdfa2V5czoKICAgICAgICAgICAgICAgIHByaW50KGYiTWlzc2luZyBrZXlzOiB7bWlzc2luZ19rZXlzfSIpCiAgICAgICAgICAgIGlmIHVuZXhwZWN0ZWRfa2V5czoKICAgICAgICAgICAgICAgIHByaW50KGYiVW5leHBlY3RlZCBrZXlzOiB7dW5leHBlY3RlZF9rZXlzfSIpCgogICAgICAgICMgTW92ZSBvbmx5IHRoZSBub24tcXVhbnRpemVkIHZlcmlmaWVyIHBhcmFtZXRlcnMgdG8gR1BVCiAgICAgICAgIyAoZ2FpbiwgYmlhcywgdnNjb3JlX2hlYWQpLiBUaGUgcXVhbnRpemVkIGJhc2VfbW9kZWwgaXMgYWxyZWFkeQogICAgICAgICMgb24gR1BVIHZpYSBkZXZpY2VfbWFwLCBidXQgY3VzdG9tIHBhcmFtcyBsb2FkZWQgZnJvbSBDUFUgbmVlZCBtb3ZpbmcuCiAgICAgICAgZGV2aWNlID0gbmV4dChiYXNlX21vZGVsLnBhcmFtZXRlcnMoKSkuZGV2aWNlCiAgICAgICAgdmVyaWZ5X21vZGVsLmdhaW4gPSB0b3JjaC5ubi5QYXJhbWV0ZXIodmVyaWZ5X21vZGVsLmdhaW4udG8oZGV2aWNlKSkKICAgICAgICB2ZXJpZnlfbW9kZWwuYmlhcyA9IHRvcmNoLm5uLlBhcmFtZXRlcih2ZXJpZnlfbW9kZWwuYmlhcy50byhkZXZpY2UpKQogICAgICAgIHZlcmlmeV9tb2RlbC52c2NvcmVfaGVhZCA9IHZlcmlmeV9tb2RlbC52c2NvcmVfaGVhZC50byhkZXZpY2UpCiAgICAgICAgdmVyaWZ5X21vZGVsLmRyb3BvdXQgPSB2ZXJpZnlfbW9kZWwuZHJvcG91dC50byhkZXZpY2UpCiAgICAgICAgdmVyaWZ5X21vZGVsLmV2YWwoKQoKICAgIHJldHVybiB2ZXJpZnlfbW9kZWwsIHRva2VuaXplcgo="
model_utils_path = f"{WORK_DIR}/traver/verifier/model_utils.py"
import pathlib
pathlib.Path(model_utils_path).write_text(base64.b64decode(_mu_b64).decode())
print("✅ Patched model_utils.py (4-bit quant, no double_quant, selective .to())")




print("\n🚀 Ready for Phase 1! Make sure to fully restart your runtime if you hit an OOM previously.")

---
## 5. Phase 1: Run Baseline TRAVER

Runs the standard Traver pipeline with the pre-trained verifier.
Progress is checkpointed — safe to restart if session disconnects.

In [ ]:
# ── Write run_mcminer_loop.py to the cloned repo ──
import base64, pathlib

_mcloop_b64 = "..."

mcloop_path = f"{WORK_DIR}/traver/run_mcminer_loop.py"
pathlib.Path(mcloop_path).write_text(
    base64.b64decode(_mcloop_b64).decode()
)
print(f"✅ Wrote run_mcminer_loop.py ({len(_mcloop_b64)} b64 chars)")
print(f"   → {mcloop_path}")


## 🔄 Phase 1: McMiner-in-the-Loop Dialogue Generation

Uses `run_mcminer_loop.py` — runs dialogue round-by-round with misconception injection.
Results saved to `{DRIVE_DIR}/output/mcminer_loop/{project}/`


In [ ]:
import subprocess, os, sys, json, shutil

# ── Setup: Create per-project prompt element files ──
os.makedirs(f"{WORK_DIR}/traver/utils", exist_ok=True)
with open(f"{WORK_DIR}/traver/__init__.py", "w") as f: pass
os.makedirs(f"{WORK_DIR}/verifier", exist_ok=True)
with open(f"{WORK_DIR}/verifier/__init__.py", "w") as f: pass
with open(f"{WORK_DIR}/traver/utils/__init__.py", "w") as f:
    f.write("from .utils import *\nfrom .make_prompt import *\n")

# Load all prompt elements
with open(f"{WORK_DIR}/prompt/prompt_elements_final.jsonl") as f:
    ALL_ELEMENTS = [json.loads(line.strip()) for line in f if line.strip()]

# Build project -> namespaces mapping
from collections import OrderedDict, defaultdict
PROJECTS = OrderedDict()
for e in ALL_ELEMENTS:
    proj = e["namespace"].split(".")[0]
    if proj not in PROJECTS:
        PROJECTS[proj] = []
    PROJECTS[proj].append(e["namespace"])

# Create per-project directories
PROJ_DIR = f"{WORK_DIR}/prompt/per_project"
os.makedirs(PROJ_DIR, exist_ok=True)

for proj, namespaces in PROJECTS.items():
    proj_dir = f"{PROJ_DIR}/{proj}"
    os.makedirs(proj_dir, exist_ok=True)

    # Write filtered prompt_elements
    proj_elements = [e for e in ALL_ELEMENTS if e["namespace"] in namespaces]
    with open(f"{proj_dir}/prompt_elements_final.jsonl", "w") as f:
        for e in proj_elements:
            f.write(json.dumps(e, ensure_ascii=False) + "\n")

    # Write namespaces.json mapping each namespace to its correct verifier part
    with open(f"{WORK_DIR}/prompt/namespaces.json") as f:
        orig_ns = json.load(f)
    ns_to_part = {}
    for pidx, part in enumerate(orig_ns["part_lists"]):
        for ns in part:
            ns_to_part[ns] = pidx

    # Group this project's namespaces by their verifier part
    part_groups = defaultdict(list)
    for ns in namespaces:
        pidx = ns_to_part.get(ns, 0)
        part_groups[pidx].append(ns)

    # Build namespaces.json with only the parts this project needs
    sorted_parts = sorted(part_groups.keys())
    ns_json = {
        "namespaces_all": namespaces,
        "num_parts": len(sorted_parts),
        "part_lists": [part_groups[p] for p in sorted_parts],
        "part_indices": sorted_parts
    }
    with open(f"{proj_dir}/namespaces.json", "w") as f:
        json.dump(ns_json, f, indent=2)

print(f"✅ Created per-project prompt files for {len(PROJECTS)} projects:")
for proj, ns_list in PROJECTS.items():
    print(f"   {proj:40s} {len(ns_list):3d} tasks")


In [ ]:
import subprocess, os, sys, json, shutil

MCMINER_OUTPUT_LOCAL = f"{WORK_DIR}/output/mcminer_loop"
MCMINER_OUTPUT_DRIVE = f"{DRIVE_DIR}/output/mcminer_loop"
os.makedirs(MCMINER_OUTPUT_LOCAL, exist_ok=True)
os.makedirs(MCMINER_OUTPUT_DRIVE, exist_ok=True)

def sync_mcminer_to_drive(project, level):
    model_name = TUTOR_MODEL_ID.split('/')[-1]
    src = f"{MCMINER_OUTPUT_LOCAL}/{project}/traver/{model_name}/{level}"
    dst = f"{MCMINER_OUTPUT_DRIVE}/{project}/traver/{model_name}/{level}"
    if not os.path.exists(src):
        return
    os.makedirs(dst, exist_ok=True)
    if os.path.realpath(src) == os.path.realpath(dst):
        print(f"   💾 {project}/{level} already on Drive (same path)")
        return
    for fn in os.listdir(src):
        try:
            shutil.copy2(os.path.join(src, fn), os.path.join(dst, fn))
        except shutil.SameFileError:
            pass
    print(f"   💾 Synced {project}/{level} to Drive")

def run_project_mcminer(project_name):
    prompt_file = f"{WORK_DIR}/prompt/per_project/{project_name}/prompt_elements_final.jsonl"
    namespace_file = f"{WORK_DIR}/prompt/per_project/{project_name}/namespaces.json"
    if not os.path.exists(prompt_file):
        print(f'⚠️  No prompt file for {project_name}')
        return
    with open(namespace_file) as f:
        ns_cfg = json.load(f)
    part_indices = ns_cfg.get('part_indices', list(range(ns_cfg['num_parts'])))
    n_tasks = len(ns_cfg['namespaces_all'])
    levels = ['low_level', 'med_level', 'high_level']
    output_dir = f"{MCMINER_OUTPUT_LOCAL}/{project_name}"

    for level in levels:
        # Check if done
        model_name = TUTOR_MODEL_ID.split('/')[-1]
        local_path = f"{output_dir}/traver/{model_name}/{level}/simulated_dialogs.jsonl"
        drive_path = f"{MCMINER_OUTPUT_DRIVE}/{project_name}/traver/{model_name}/{level}/simulated_dialogs.jsonl"
        if not os.path.exists(local_path) and os.path.exists(drive_path):
            os.makedirs(os.path.dirname(local_path), exist_ok=True)
            shutil.copy2(drive_path, local_path)
        if os.path.exists(local_path):
            with open(local_path) as f:
                done = sum(1 for l in f if l.strip())
            if done >= n_tasks:
                print(f'  ✅ {project_name}/{level}: already complete ({done}/{n_tasks})')
                continue

        for part_idx in part_indices:
            single_ns = ns_cfg['part_lists'][part_indices.index(part_idx)]
            single_ns_file = f"{WORK_DIR}/prompt/per_project/{project_name}/namespaces_mcloop_p{part_idx}.json"
            with open(single_ns_file, 'w') as f:
                json.dump({'namespaces_all': single_ns, 'num_parts': 1,
                           'part_lists': [single_ns], 'part_indices': [part_idx]}, f)

            print(f'\n🔄 {project_name}/{level} part {part_idx} ({len(single_ns)} tasks)')
            env = os.environ.copy()
            env['PYTHONPATH'] = f"{WORK_DIR}/traver:{WORK_DIR}"
            env['PYTHONUNBUFFERED'] = '1'
            env['HF_TOKEN'] = HF_TOKEN

            cmd = [
                'python', f'{WORK_DIR}/traver/run_mcminer_loop.py',
                '--namespace_file', single_ns_file,
                '--prompt_element_file', prompt_file,
                '--output_dir', output_dir,
                '--student_setting', level,
                '--tutor_model_name_or_path', TUTOR_MODEL_ID,
                '--student_model_name_or_path', STUDENT_MODEL_ID,
                '--codegen_model', 'meta-llama/Llama-3.1-8B-Instruct',
                '--codegen_api_base', HF_API_BASE,
                '--tutor_num_responses', str(TUTOR_NUM_RESPONSES),
                '--vllm_api_key', HF_TOKEN,
                '--vllm_endpoint_tutor', HF_API_BASE,
                '--vllm_endpoint_student', 'local',
                '--verifier_base_model_path', f'{MODEL_DIR}/Mistral-7B-v0.1',
                '--verifier_model_dir', f'{MODEL_DIR}/Verifier-7B',
                '--verifier_model_parts', str(part_idx),
                '--google_api_key', GOOGLE_API_KEY,
                '--show_message', 'true',
            ]

            proc = subprocess.Popen(cmd, env=env, cwd=WORK_DIR,
                                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                     text=True, bufsize=1)
            for line in iter(proc.stdout.readline, ''):
                print(line, end='')
                sys.stdout.flush()
            proc.stdout.close()
            proc.wait()

        sync_mcminer_to_drive(project_name, level)
    print(f'\n✅ {project_name} McMiner-loop dialogue complete!')

print('🚀 run_project_mcminer() ready.')


### 📦 easyvolcap → EasyVolcap (12 tasks) — McMiner Loop


In [ ]:
run_project_mcminer('easyvolcap')


### 📦 searcharray → searcharray (6 tasks) — McMiner Loop


In [ ]:
run_project_mcminer('searcharray')


### 📦 xinhua → UHGEval (1 task) — McMiner Loop


In [ ]:
run_project_mcminer('xinhua')


### 📦 gfpgan_model + codeformer_model → sd-forge (3 tasks) — McMiner Loop


In [ ]:
run_project_mcminer('gfpgan_model')
run_project_mcminer('codeformer_model')


---
## 📊 Phase 2: Evaluate McMiner-Loop Dialogues

Generate N=10 completions per round, run tests, compute Pass@k.
Uses the SAME evaluation pipeline as baseline — only the dialogue source changes.


In [ ]:
import base64, pathlib

# ── Write LM_inference_hf.py (not in the cloned repo) ──
_lm_hf_b64 = "IiIiCkFQSS1iYXNlZCBMTSBpbmZlcmVuY2UgZm9yIEhQQyAobm8gbG9jYWwgbW9kZWwgbmVlZGVkKS4KRHJvcC1pbiByZXBsYWNlbWVudCBmb3IgTE1faW5mZXJlbmNlLnB5IHRoYXQgdXNlcyBhbiBPcGVuQUktY29tcGF0aWJsZSBBUEkKKEdyb3EsIFRvZ2V0aGVyIEFJLCBIdWdnaW5nRmFjZSwgZXRjLikgaW5zdGVhZCBvZiBsb2FkaW5nIHRoZSBtb2RlbCBsb2NhbGx5LgoKVXNlcyBhc3luY2lvIGZvciBjb25jdXJyZW50IEFQSSBjYWxsczoKICAtIEFsbCBOIGNvbXBsZXRpb25zIHBlciB0YXNrIGZpcmUgc2ltdWx0YW5lb3VzbHkKICAtIFVwIHRvIC0tbWF4X2NvbmN1cnJlbnRfdGFza3MgdGFza3MgcHJvY2VzcyBpbiBwYXJhbGxlbAogIC0gRXhwb25lbnRpYWwgYmFja29mZiB3aXRoIGppdHRlciBvbiByYXRlLWxpbWl0ICg0MjkpIGVycm9ycwoKVXNhZ2U6CiAgICBweXRob24gdHJhdmVyL3V0aWxzL0xNX2luZmVyZW5jZV9oZi5weSBcCiAgICAgICAgLS1wcm9tcHRfZmlsZSBwcm9tcHQuanNvbmwgXAogICAgICAgIC0tb3V0cHV0X2RpciBvdXRwdXQvIFwKICAgICAgICAtLW1vZGVsX25hbWVfb3JfcGF0aCBtZXRhLWxsYW1hL0xsYW1hLTMuMS04Qi1JbnN0cnVjdCBcCiAgICAgICAgLS1hcGlfYmFzZSBodHRwczovL2FwaS5ncm9xLmNvbS9vcGVuYWkvdjEgXAogICAgICAgIC0tYXBpX2tleSBnc2tfeHh4IFwKICAgICAgICAtLW1heF9jb25jdXJyZW50X3Rhc2tzIDUKIiIiCgppbXBvcnQgYXN5bmNpbwppbXBvcnQganNvbgppbXBvcnQgb3MKaW1wb3J0IHJhbmRvbQppbXBvcnQgcmUKaW1wb3J0IHRpbWUKZnJvbSB0cWRtIGltcG9ydCB0cWRtCmZyb20gYXJncGFyc2UgaW1wb3J0IEFyZ3VtZW50UGFyc2VyCmZyb20gb3BlbmFpIGltcG9ydCBBc3luY09wZW5BSQoKCmRlZiBwYXJzZV9hcmdzKCk6CiAgICBwYXJzZXIgPSBBcmd1bWVudFBhcnNlcigpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCctLXByb21wdF9maWxlJywgdHlwZT1zdHIsIHJlcXVpcmVkPVRydWUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW91dHB1dF9kaXIiLCB0eXBlPXN0ciwgcmVxdWlyZWQ9VHJ1ZSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tbW9kZWxfbmFtZV9vcl9wYXRoIiwgdHlwZT1zdHIsCiAgICAgICAgICAgICAgICAgICAgICAgIGRlZmF1bHQ9Im1ldGEtbGxhbWEvTGxhbWEtMy4xLThCLUluc3RydWN0IikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoJy0tZGVjb2RpbmcnLCB0eXBlPXN0ciwgZGVmYXVsdD0nc2FtcGxpbmcnLAogICAgICAgICAgICAgICAgICAgICAgICBjaG9pY2VzPVsnZ3JlZWR5JywgJ3NhbXBsaW5nJ10pCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW1heF90b2tlbnMiLCB0eXBlPWludCwgZGVmYXVsdD01MDApCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCctLVQnLCB0eXBlPWZsb2F0LCBkZWZhdWx0PTAuNCkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoJy0tdG9wX3AnLCB0eXBlPWZsb2F0LCBkZWZhdWx0PTAuOTUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCctLU4nLCB0eXBlPWludCwgZGVmYXVsdD0xMCkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0teWVzX29yX25vX3JlcXVpcmVkIiwgYWN0aW9uPSdzdG9yZV90cnVlJykKCiAgICAjIEFQSSBjb25maWcKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoJy0tYXBpX2Jhc2UnLCB0eXBlPXN0ciwKICAgICAgICAgICAgICAgICAgICAgICAgZGVmYXVsdD1vcy5lbnZpcm9uLmdldCgnTExBTUFfQVBJX0VORFBPSU5UJywKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAnaHR0cHM6Ly9hcGkuZ3JvcS5jb20vb3BlbmFpL3YxJykpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCctLWFwaV9rZXknLCB0eXBlPXN0ciwKICAgICAgICAgICAgICAgICAgICAgICAgZGVmYXVsdD1vcy5lbnZpcm9uLmdldCgnR1JPUV9BUElfS0VZJywgJycpKQoKICAgICMgQXN5bmMgY29uY3VycmVuY3kgY29uZmlnCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCctLW1heF9jb25jdXJyZW50X3Rhc2tzJywgdHlwZT1pbnQsIGRlZmF1bHQ9NSwKICAgICAgICAgICAgICAgICAgICAgICAgaGVscD0iTWF4IG51bWJlciBvZiB0YXNrcyB0byBwcm9jZXNzIGluIHBhcmFsbGVsIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoJy0tbWF4X3JldHJpZXMnLCB0eXBlPWludCwgZGVmYXVsdD04LAogICAgICAgICAgICAgICAgICAgICAgICBoZWxwPSJNYXggcmV0cmllcyBwZXIgQVBJIGNhbGwgb24gcmF0ZS1saW1pdCBlcnJvcnMiKQoKICAgIHJldHVybiBwYXJzZXIucGFyc2VfYXJncygpCgoKZGVmIGxvYWRfZmluaXNoZWRfZGF0YShvdXRwdXRfZmlsZSk6CiAgICBmaW5pc2hlZCA9IHNldCgpCiAgICBpZiBvcy5wYXRoLmV4aXN0cyhvdXRwdXRfZmlsZSk6CiAgICAgICAgd2l0aCBvcGVuKG91dHB1dF9maWxlLCAncicpIGFzIGY6CiAgICAgICAgICAgIGZvciBsaW5lIGluIGY6CiAgICAgICAgICAgICAgICBqcyA9IGpzb24ubG9hZHMobGluZSkKICAgICAgICAgICAgICAgIGZpbmlzaGVkLmFkZChqc1snbmFtZXNwYWNlJ10pCiAgICByZXR1cm4gZmluaXNoZWQKCgphc3luYyBkZWYgYXN5bmNfc2luZ2xlX2NvbXBsZXRpb24oY2xpZW50LCBtb2RlbCwgcHJvbXB0LCB0ZW1wZXJhdHVyZSwgdG9wX3AsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBtYXhfdG9rZW5zLCBtYXhfcmV0cmllcyk6CiAgICAiIiJNYWtlIGEgc2luZ2xlIEFQSSBjYWxsIHdpdGggZXhwb25lbnRpYWwgYmFja29mZiBvbiByYXRlLWxpbWl0IGVycm9ycy4iIiIKICAgIGZvciBhdHRlbXB0IGluIHJhbmdlKG1heF9yZXRyaWVzKToKICAgICAgICB0cnk6CiAgICAgICAgICAgIHJlc3BvbnNlID0gYXdhaXQgY2xpZW50LmNoYXQuY29tcGxldGlvbnMuY3JlYXRlKAogICAgICAgICAgICAgICAgbW9kZWw9bW9kZWwsCiAgICAgICAgICAgICAgICBtZXNzYWdlcz1beyJyb2xlIjogInVzZXIiLCAiY29udGVudCI6IHByb21wdH1dLAogICAgICAgICAgICAgICAgdGVtcGVyYXR1cmU9dGVtcGVyYXR1cmUsCiAgICAgICAgICAgICAgICB0b3BfcD10b3BfcCwKICAgICAgICAgICAgICAgIG1heF90b2tlbnM9bWF4X3Rva2VucywKICAgICAgICAgICAgKQogICAgICAgICAgICByZXR1cm4gcmVzcG9uc2UuY2hvaWNlc1swXS5tZXNzYWdlLmNvbnRlbnQKICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6CiAgICAgICAgICAgIGVycm9yX3N0ciA9IHN0cihlKS5sb3dlcigpCiAgICAgICAgICAgIGlmICc0MjknIGluIGVycm9yX3N0ciBvciAncmF0ZScgaW4gZXJyb3Jfc3RyOgogICAgICAgICAgICAgICAgIyBFeHBvbmVudGlhbCBiYWNrb2ZmIHdpdGggaml0dGVyCiAgICAgICAgICAgICAgICB3YWl0ID0gbWluKDIgKiogYXR0ZW1wdCArIHJhbmRvbS51bmlmb3JtKDAsIDEpLCA2MCkKICAgICAgICAgICAgICAgIHByaW50KGYiICBSYXRlIGxpbWl0ZWQgKGF0dGVtcHQge2F0dGVtcHQgKyAxfS97bWF4X3JldHJpZXN9KSwgIgogICAgICAgICAgICAgICAgICAgICAgZiJ3YWl0aW5nIHt3YWl0Oi4xZn1zLi4uIikKICAgICAgICAgICAgICAgIGF3YWl0IGFzeW5jaW8uc2xlZXAod2FpdCkKICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgIHByaW50KGYiICBBUEkgZXJyb3I6IHtlfSIpCiAgICAgICAgICAgICAgICByZXR1cm4gIiIKICAgIHByaW50KGYiICBNYXggcmV0cmllcyAoe21heF9yZXRyaWVzfSkgZXhjZWVkZWQsIHJldHVybmluZyBlbXB0eSBzdHJpbmciKQogICAgcmV0dXJuICIiCgoKYXN5bmMgZGVmIGFzeW5jX2FwaV9nZW5lcmF0ZShjbGllbnQsIG1vZGVsLCBwcm9tcHQsIG4sIHRlbXBlcmF0dXJlLCB0b3BfcCwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgbWF4X3Rva2VucywgbWF4X3JldHJpZXMpOgogICAgIiIiR2VuZXJhdGUgbiBjb21wbGV0aW9ucyBjb25jdXJyZW50bHkgdmlhIGFzeW5jaW8uZ2F0aGVyKCkuIiIiCiAgICB0YXNrcyA9IFsKICAgICAgICBhc3luY19zaW5nbGVfY29tcGxldGlvbihjbGllbnQsIG1vZGVsLCBwcm9tcHQsIHRlbXBlcmF0dXJlLCB0b3BfcCwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBtYXhfdG9rZW5zLCBtYXhfcmV0cmllcykKICAgICAgICBmb3IgXyBpbiByYW5nZShuKQogICAgXQogICAgcmV0dXJuIGF3YWl0IGFzeW5jaW8uZ2F0aGVyKCp0YXNrcykKCgphc3luYyBkZWYgYXN5bmNfeWVzX29yX25vKGNsaWVudCwgbW9kZWwsIHByb21wdCwgdGVtcGVyYXR1cmUsIHRvcF9wLAogICAgICAgICAgICAgICAgICAgICAgICAgICBtYXhfdG9rZW5zLCBtYXhfcmV0cmllcyk6CiAgICAiIiJIYW5kbGUgeWVzL25vIGNsYXNzaWZpY2F0aW9uIHdpdGggc2VxdWVudGlhbCByZXRyaWVzLiIiIgogICAgbWF4X3RyaWVzID0gMwogICAgZm9yIGF0dGVtcHQgaW4gcmFuZ2UobWF4X3RyaWVzKToKICAgICAgICByZXN1bHQgPSBhd2FpdCBhc3luY19zaW5nbGVfY29tcGxldGlvbigKICAgICAgICAgICAgY2xpZW50LCBtb2RlbCwgcHJvbXB0LCB0ZW1wZXJhdHVyZSwgdG9wX3AsIG1heF90b2tlbnMsIG1heF9yZXRyaWVzCiAgICAgICAgKQogICAgICAgIHRleHQgPSByZXN1bHQuc3RyaXAoKS5sb3dlcigpIGlmIHJlc3VsdCBlbHNlICIiCiAgICAgICAgaWYgcmUubWF0Y2gociJ5ZXN8eXx5ZWF8eWVhaHx5ZXB8eXVwfHN1cmV8b2t8b2theXxhbHJpZ2h0IiwKICAgICAgICAgICAgICAgICAgICB0ZXh0LCByZS5JR05PUkVDQVNFKToKICAgICAgICAgICAgcmV0dXJuICJ5ZXMiCiAgICAgICAgZWxpZiByZS5tYXRjaChyIm5vfG58bm9wZXxuYWh8bmF5IiwgdGV4dCwgcmUuSUdOT1JFQ0FTRSk6CiAgICAgICAgICAgIHJldHVybiAibm8iCiAgICByZXR1cm4gIm5vIgoKCmFzeW5jIGRlZiBwcm9jZXNzX3Rhc2soc2VtYXBob3JlLCBjbGllbnQsIGFyZ3MsIGpzLCBmX291dCwgcGJhcik6CiAgICAiIiJQcm9jZXNzIGEgc2luZ2xlIHRhc2sgKGFsbCBOIGNvbXBsZXRpb25zKSB1bmRlciB0aGUgc2VtYXBob3JlLiIiIgogICAgYXN5bmMgd2l0aCBzZW1hcGhvcmU6CiAgICAgICAgcHJvbXB0ID0ganNbJ3Byb21wdCddCiAgICAgICAgdGFza19pZCA9IGpzWyduYW1lc3BhY2UnXQoKICAgICAgICBpZiBhcmdzLnllc19vcl9ub19yZXF1aXJlZDoKICAgICAgICAgICAgY29tcGxldGlvbnMgPSBhd2FpdCBhc3luY195ZXNfb3Jfbm8oCiAgICAgICAgICAgICAgICBjbGllbnQsIGFyZ3MubW9kZWxfbmFtZV9vcl9wYXRoLCBwcm9tcHQsCiAgICAgICAgICAgICAgICBhcmdzLlQsIGFyZ3MudG9wX3AsIGFyZ3MubWF4X3Rva2VucywgYXJncy5tYXhfcmV0cmllcwogICAgICAgICAgICApCiAgICAgICAgZWxzZToKICAgICAgICAgICAgcmVzdWx0cyA9IGF3YWl0IGFzeW5jX2FwaV9nZW5lcmF0ZSgKICAgICAgICAgICAgICAgIGNsaWVudCwgYXJncy5tb2RlbF9uYW1lX29yX3BhdGgsIHByb21wdCwKICAgICAgICAgICAgICAgIGFyZ3MuTiwgYXJncy5ULCBhcmdzLnRvcF9wLCBhcmdzLm1heF90b2tlbnMsIGFyZ3MubWF4X3JldHJpZXMKICAgICAgICAgICAgKQogICAgICAgICAgICBjb21wbGV0aW9ucyA9IGxpc3QocmVzdWx0cykKCiAgICAgICAgY2FzZXMgPSB7J25hbWVzcGFjZSc6IHRhc2tfaWQsICdjb21wbGV0aW9uJzogY29tcGxldGlvbnN9CiAgICAgICAgZl9vdXQud3JpdGUoanNvbi5kdW1wcyhjYXNlcykgKyAnXG4nKQogICAgICAgIGZfb3V0LmZsdXNoKCkKICAgICAgICBwYmFyLnVwZGF0ZSgxKQoKCmFzeW5jIGRlZiBhc3luY19tYWluKGFyZ3MsIHRvZG9fdGFza3MpOgogICAgIiIiTWFpbiBhc3luYyBlbnRyeSBwb2ludDogcHJvY2VzcyBhbGwgdGFza3Mgd2l0aCBib3VuZGVkIGNvbmN1cnJlbmN5LiIiIgogICAgY2xpZW50ID0gQXN5bmNPcGVuQUkoCiAgICAgICAgYXBpX2tleT1hcmdzLmFwaV9rZXksCiAgICAgICAgYmFzZV91cmw9YXJncy5hcGlfYmFzZQogICAgKQoKICAgIHNlbWFwaG9yZSA9IGFzeW5jaW8uU2VtYXBob3JlKGFyZ3MubWF4X2NvbmN1cnJlbnRfdGFza3MpCiAgICBvdXRwdXRfZmlsZSA9IG9zLnBhdGguam9pbihhcmdzLm91dHB1dF9kaXIsICdjb21wbGV0aW9uX2xtLmpzb25sJykKCiAgICB3aXRoIG9wZW4ob3V0cHV0X2ZpbGUsICdhJykgYXMgZl9vdXQ6CiAgICAgICAgd2l0aCB0cWRtKHRvdGFsPWxlbih0b2RvX3Rhc2tzKSwgZGVzYz0iR2VuZXJhdGluZyIpIGFzIHBiYXI6CiAgICAgICAgICAgIHRhc2tzID0gWwogICAgICAgICAgICAgICAgcHJvY2Vzc190YXNrKHNlbWFwaG9yZSwgY2xpZW50LCBhcmdzLCBqcywgZl9vdXQsIHBiYXIpCiAgICAgICAgICAgICAgICBmb3IganMgaW4gdG9kb190YXNrcwogICAgICAgICAgICBdCiAgICAgICAgICAgIGF3YWl0IGFzeW5jaW8uZ2F0aGVyKCp0YXNrcykKCgpkZWYgbWFpbigpOgogICAgYXJncyA9IHBhcnNlX2FyZ3MoKQoKICAgIGlmIG5vdCBhcmdzLmFwaV9rZXk6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigKICAgICAgICAgICAgIkFQSSBrZXkgcmVxdWlyZWQuIFNldCBMTEFNQV9BUElfS0VZIGVudiB2YXIgb3IgdXNlIC0tYXBpX2tleSIpCgogICAgaWYgYXJncy55ZXNfb3Jfbm9fcmVxdWlyZWQ6CiAgICAgICAgYXJncy5OID0gMQogICAgaWYgYXJncy5kZWNvZGluZyA9PSAnZ3JlZWR5JzoKICAgICAgICBhcmdzLlQgPSAwCiAgICAgICAgYXJncy50b3BfcCA9IDEKICAgICAgICBhcmdzLk4gPSAxCgogICAgcHJpbnQoZiJNb2RlbDogICAge2FyZ3MubW9kZWxfbmFtZV9vcl9wYXRofSIpCiAgICBwcmludChmIkFQSSBiYXNlOiB7YXJncy5hcGlfYmFzZX0iKQogICAgcHJpbnQoZiJOOiAgICAgICAge2FyZ3MuTn0iKQogICAgcHJpbnQoZiJUZW1wOiAgICAge2FyZ3MuVH0iKQogICAgcHJpbnQoZiJDb25jdXJyZW5jeToge2FyZ3MubWF4X2NvbmN1cnJlbnRfdGFza3N9IHRhc2tzIGluIHBhcmFsbGVsIikKCiAgICAjIFZlcmlmeSBjb25uZWN0aXZpdHkgKHN5bmNocm9ub3VzLCBvbmUtb2ZmIGNoZWNrKQogICAgZnJvbSBvcGVuYWkgaW1wb3J0IE9wZW5BSQogICAgc3luY19jbGllbnQgPSBPcGVuQUkoYXBpX2tleT1hcmdzLmFwaV9rZXksIGJhc2VfdXJsPWFyZ3MuYXBpX2Jhc2UpCiAgICB0cnk6CiAgICAgICAgbW9kZWxzID0gc3luY19jbGllbnQubW9kZWxzLmxpc3QoKQogICAgICAgIGF2YWlsYWJsZSA9IFttLmlkIGZvciBtIGluIG1vZGVscy5kYXRhXQogICAgICAgIHByaW50KGYiQXZhaWxhYmxlIG1vZGVsczoge2F2YWlsYWJsZVs6NV19Li4uIikKICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToKICAgICAgICBwcmludChmIuKaoCAgQ291bGQgbm90IGxpc3QgbW9kZWxzOiB7ZX0iKQoKICAgIGlmIG5vdCBvcy5wYXRoLmV4aXN0cyhhcmdzLm91dHB1dF9kaXIpOgogICAgICAgIG9zLm1ha2VkaXJzKGFyZ3Mub3V0cHV0X2RpcikKCiAgICBvdXRwdXRfZmlsZSA9IG9zLnBhdGguam9pbihhcmdzLm91dHB1dF9kaXIsICdjb21wbGV0aW9uX2xtLmpzb25sJykKICAgIGZpbmlzaGVkX2RhdGEgPSBsb2FkX2ZpbmlzaGVkX2RhdGEob3V0cHV0X2ZpbGUpCiAgICBwcmludChmIlNraXBwaW5nIHtsZW4oZmluaXNoZWRfZGF0YSl9IGFscmVhZHkgY29tcGxldGVkIHRhc2tzIikKCiAgICBpZiBub3Qgb3MucGF0aC5leGlzdHMoYXJncy5wcm9tcHRfZmlsZSk6CiAgICAgICAgcHJpbnQoZiJQcm9tcHQgZmlsZSBub3QgZm91bmQ6IHthcmdzLnByb21wdF9maWxlfSIpCiAgICAgICAgcmV0dXJuCgogICAgIyBMb2FkIGFsbCB0YXNrcyBhbmQgZmlsdGVyIG91dCBmaW5pc2hlZCBvbmVzCiAgICB0b2RvX3Rhc2tzID0gW10KICAgIHdpdGggb3BlbihhcmdzLnByb21wdF9maWxlLCAncicpIGFzIGZfaW46CiAgICAgICAgZm9yIGxpbmUgaW4gZl9pbjoKICAgICAgICAgICAganMgPSBqc29uLmxvYWRzKGxpbmUpCiAgICAgICAgICAgIGlmIGpzWyduYW1lc3BhY2UnXSBub3QgaW4gZmluaXNoZWRfZGF0YToKICAgICAgICAgICAgICAgIHRvZG9fdGFza3MuYXBwZW5kKGpzKQoKICAgIHByaW50KGYiVE9ETyB0YXNrczoge2xlbih0b2RvX3Rhc2tzKX0iKQoKICAgIGlmIG5vdCB0b2RvX3Rhc2tzOgogICAgICAgIHByaW50KCJBbGwgdGFza3MgYWxyZWFkeSBjb21wbGV0ZWQhIikKICAgICAgICByZXR1cm4KCiAgICBzdGFydCA9IHRpbWUudGltZSgpCiAgICBhc3luY2lvLnJ1bihhc3luY19tYWluKGFyZ3MsIHRvZG9fdGFza3MpKQogICAgZWxhcHNlZCA9IHRpbWUudGltZSgpIC0gc3RhcnQKCiAgICBwcmludChmIkRvbmUuIE91dHB1dDoge291dHB1dF9maWxlfSIpCiAgICBwcmludChmIldhbGwgdGltZToge2VsYXBzZWQ6LjFmfXMgKHtlbGFwc2VkLzYwOi4xZn0gbWluKSIpCgoKaWYgX19uYW1lX18gPT0gJ19fbWFpbl9fJzoKICAgIG1haW4oKQo="
_lm_hf_path = f"{WORK_DIR}/traver/utils/LM_inference_hf.py"
pathlib.Path(_lm_hf_path).write_bytes(base64.b64decode(_lm_hf_b64))
print(f"✅ Wrote LM_inference_hf.py ({len(_lm_hf_b64)} b64 chars)")

# ── Install tree-sitter for process_completion.py ──
import subprocess as _sp
_sp.run(["pip", "install", "-q", "tree-sitter==0.20.4"], check=True)
import os
if not os.path.exists(f"{WORK_DIR}/build/my-languages.so"):
    _sp.run(["git", "clone", "-q", "https://github.com/tree-sitter/tree-sitter-python",
             f"{WORK_DIR}/vendor/tree-sitter-python"], check=True)
    os.makedirs(f"{WORK_DIR}/build", exist_ok=True)
    from tree_sitter import Language
    Language.build_library(f"{WORK_DIR}/build/my-languages.so",
                          [f"{WORK_DIR}/vendor/tree-sitter-python"])
    print("✅ Built tree-sitter Python grammar")
else:
    print("✅ tree-sitter already set up")

# ── Install eval dependencies ──
_sp.run(["pip", "install", "-q", "func_timeout", "psutil"], check=True)
print("✅ Installed func_timeout, psutil")


import subprocess, os, sys, json, shutil, glob, time
import numpy as np
from collections import defaultdict

# ── Config ──
POSTTEST_N = 10
POSTTEST_MAX_TOKENS = 1024
POSTTEST_TEMP = 0.4
POSTTEST_TOP_P = 0.95
MAX_INTERACTION_ROUND = 8
MAX_CONCURRENT_TASKS = 3
CODEGEN_MODEL = "meta-llama/Llama-3.1-8B-Instruct"
CODEGEN_API_BASE = "https://router.huggingface.co/v1"

model_name = TUTOR_MODEL_ID.split("/")[-1]
# Copy source code to LOCAL storage (Drive has write latency that breaks pass_k)
_drive_source = f"{DATA_DIR}/Tutor-Agents/Source_Code"
Source_Code_Root = f"{WORK_DIR}/Source_Code_Local"
os.makedirs(Source_Code_Root, exist_ok=True)
# Only copy the 3 projects we need (not all 10 — some are huge)
_EVAL_PROJECTS = ["UHGEval", "searcharray", "stable-diffusion-webui-forge"]
import shutil as _sh
for _p in _EVAL_PROJECTS:
    _dst = f"{Source_Code_Root}/{_p}"
    if not os.path.exists(_dst):
        _src = f"{_drive_source}/{_p}"
        if os.path.exists(_src):
            _sh.copytree(_src, _dst)
            print(f"  ✅ Copied {_p} to local")
    else:
        print(f"  ✅ {_p} already local")
_drive_dep = f"{DATA_DIR}/Tutor-Agents/Dependency_Data"
Dependency_Root = f"{WORK_DIR}/Dependency_Data_Local"
os.makedirs(Dependency_Root, exist_ok=True)
for _p in _EVAL_PROJECTS:
    _dst = f"{Dependency_Root}/{_p}"
    _src = f"{_drive_dep}/{_p}"
    if not os.path.exists(_dst) and os.path.exists(_src):
        _sh.copytree(_src, _dst)
metadata_file = f"{WORK_DIR}/benchmark/EvoCodeBench-2403/metadata.jsonl"

# Namespace → Source Code Folder mapping
NS_TO_FOLDER = {
    "easyvolcap": "EasyVolcap",
    "searcharray": "searcharray",
    "xinhua": "UHGEval",
    "gfpgan_model": "stable-diffusion-webui-forge",
    "codeformer_model": "stable-diffusion-webui-forge",
}

EVAL_DEPS = "numpy tqdm psutil func_timeout tree_sitter pytest pytest-runner dill jinja2"


def compute_pass_at_k(n, c, k):
    if n - c < k: return 1.0
    return 1.0 - np.prod(1.0 - k / np.arange(n - c + 1, n + 1))


def run_full_pipeline(ns_prefix, levels=None):
    """Run the FULL pipeline for a namespace prefix:
       make_prompt → codegen → parse → pass@k + recall@k
    
    Args:
        ns_prefix: Namespace prefix (e.g. 'searcharray', 'xinhua', 'gfpgan_model')
    """
    if levels is None:
        levels = STUDENT_LEVELS
    
    folder = NS_TO_FOLDER.get(ns_prefix, ns_prefix)
    proj_prompt_file = f"{PROJ_DIR}/{ns_prefix}/prompt_elements_final.jsonl"
    proj_source = f"{Source_Code_Root}/{folder}"
    
    if not os.path.exists(proj_prompt_file):
        print(f"❌ No prompt file for {ns_prefix}")
        return
    
    env = os.environ.copy()
    env["PYTHONPATH"] = f"{WORK_DIR}/traver:{WORK_DIR}"
    env["PYTHONUNBUFFERED"] = "1"
    env["HF_TOKEN"] = HF_TOKEN
    
    start_time = time.time()
    
    for level in levels:
        print(f"\n{'='*65}")
        print(f"  {ns_prefix} → {folder} | {level}")
        print(f"{'='*65}")
        
        # ── Find dialogue file from Drive ──
        dialog_file = None
        for base in [MCMINER_OUTPUT_LOCAL, MCMINER_OUTPUT_DRIVE]:
            path = f"{base}/{ns_prefix}/traver/{model_name}/{level}/simulated_dialogs.jsonl"
            if os.path.exists(path):
                dialog_file = path
                break
        
        if not dialog_file:
            print(f"  ⏭️  No dialogue file, skipping")
            continue
        
        with open(dialog_file) as f:
            n_dialogs = sum(1 for line in f if line.strip())
        print(f"  📂 {n_dialogs} dialogues loaded")
        
        out_dir = f"{WORK_DIR}/output/student_posttest_mcminer/{ns_prefix}/traver/{model_name}/{level}"
        os.makedirs(out_dir, exist_ok=True)
        
        # ══════════════════════════════════════════════════
        # STEP 1: Generate posttest prompts
        # ══════════════════════════════════════════════════
        prompt_check = f"{out_dir}/prompt_round_1.jsonl"
        if os.path.exists(prompt_check):
            print(f"  ✅ Step 1: Prompts already generated")
        else:
            print(f"  📝 Step 1: Generating posttest prompts...")
            cmd = [
                "python", f"{WORK_DIR}/traver/utils/make_prompt.py",
                "--student_posttest",
                "--prompt_element_file", proj_prompt_file,
                "--simulated_file", dialog_file,
                "--output_dir", out_dir,
                "--student_level", level,
                "--max_interaction_round", str(MAX_INTERACTION_ROUND),
            ]
            r = subprocess.run(cmd, env=env, cwd=WORK_DIR, capture_output=True, text=True)
            if r.returncode != 0:
                print(f"  ❌ make_prompt failed: {r.stderr[-300:]}")
                continue
            print(f"  ✅ Step 1: Prompts for {MAX_INTERACTION_ROUND} rounds")
        
        # ══════════════════════════════════════════════════
        # STEP 2+3: Code generation + parse (per round)
        # ══════════════════════════════════════════════════
        for rdx in range(1, MAX_INTERACTION_ROUND + 1):
            round_prompt = f"{out_dir}/prompt_round_{rdx}.jsonl"
            round_dir = f"{out_dir}/round_{rdx}"
            comp_lm = f"{round_dir}/completion_lm.jsonl"
            comp_parsed = f"{round_dir}/completion.jsonl"
            
            if not os.path.exists(round_prompt):
                continue
            with open(round_prompt) as f:
                n_tasks = sum(1 for line in f if line.strip())
            if n_tasks == 0:
                continue
            
            os.makedirs(round_dir, exist_ok=True)
            
            # Skip if fully done
            if os.path.exists(comp_parsed):
                with open(comp_parsed) as f:
                    if sum(1 for l in f if l.strip()) >= n_tasks * POSTTEST_N:
                        continue
            
            # Step 2: LM inference
            need_lm = True
            if os.path.exists(comp_lm):
                with open(comp_lm) as f:
                    n_done = sum(1 for l in f if l.strip())
                if n_done >= n_tasks:
                    need_lm = False
                else:
                    print(f"  ⏩ Round {rdx}: Resuming codegen ({n_done}/{n_tasks})")
            
            if need_lm:
                print(f"  🔧 Round {rdx}: Generating {n_tasks}×{POSTTEST_N} completions...")
                cmd = [
                    "python", f"{WORK_DIR}/traver/utils/LM_inference_hf.py",
                    "--prompt_file", round_prompt,
                    "--output_dir", round_dir,
                    "--model_name_or_path", CODEGEN_MODEL,
                    "--api_base", CODEGEN_API_BASE,
                    "--api_key", HF_TOKEN,
                    "--N", str(POSTTEST_N),
                    "--T", str(POSTTEST_TEMP),
                    "--top_p", str(POSTTEST_TOP_P),
                    "--max_tokens", str(POSTTEST_MAX_TOKENS),
                    "--max_concurrent_tasks", str(MAX_CONCURRENT_TASKS),
                ]
                proc = subprocess.run(cmd, env=env, cwd=WORK_DIR,
                                     capture_output=True, text=True, timeout=1800)
                if proc.returncode != 0:
                    print(f"  ⚠️  Round {rdx} codegen failed (exit code {proc.returncode})")
                    print(f"  STDOUT (last 500 chars):")
                    print(proc.stdout[-500:] if proc.stdout else "(empty)")
                    print(f"  STDERR (last 500 chars):")
                    print(proc.stderr[-500:] if proc.stderr else "(empty)")
                    continue
            
            # Step 3: Parse completions
            if not os.path.exists(comp_parsed):
                print(f"  📋 Round {rdx}: Parsing completions...")
                r = subprocess.run([
                    "python", f"{WORK_DIR}/traver/utils/process_completion.py",
                    "--model_type", "gpt",
                    "--completion_file", comp_lm,
                    "--output_file", comp_parsed,
                ], env=env, cwd=WORK_DIR, capture_output=True, text=True)
                if r.returncode != 0:
                    print(f"  ⚠️  Round {rdx} parse failed: {r.stderr[-200:]}")
                else:
                    print(f"  ✅ Round {rdx}: {r.stdout.strip()}")
        
        # ══════════════════════════════════════════════════
        # STEP 4: Evaluation (pass@k + recall@k)
        # ══════════════════════════════════════════════════
        if not os.path.exists(proj_source):
            print(f"\n  ⚠️  Source code not at {proj_source}, skipping eval")
            continue
        
        # Filter metadata for this namespace
        filtered_data = f"{out_dir}/data_filtered.jsonl"
        # Delete stale filtered data (empty or wrong schema)
        if os.path.exists(filtered_data):
            with open(filtered_data) as _f:
                _lines = _f.readlines()
            if len(_lines) == 0 or ('completion_path' not in _lines[0]):
                os.remove(filtered_data)
                print(f"  🔄 Removed stale data_filtered.jsonl ({len(_lines)} entries, wrong schema)")
        if not os.path.exists(filtered_data):
            with open(metadata_file) as fin, open(filtered_data, 'w') as fout:
                count = 0
                for line in fin:
                    js = json.loads(line)
                    ns = js['namespace']
                    ns_p = ns.split('.')[0]
                    if ns_p == ns_prefix:
                        fout.write(line)
                        count += 1
                print(f"  📋 Filtered {count} eval tasks for {ns_prefix}")
        
        print(f"\n  🧪 Running evaluation...")
        for rdx in range(1, MAX_INTERACTION_ROUND + 1):
            comp_file = f"{out_dir}/round_{rdx}/completion.jsonl"
            log_dir = f"{out_dir}/round_{rdx}"
            test_log = f"{log_dir}/test_results.jsonl"
            
            if not os.path.exists(comp_file):
                continue
            with open(comp_file) as f:
                n_comp = sum(1 for l in f if l.strip())
            if n_comp == 0:
                continue
            
            # Skip if already evaluated
            if os.path.exists(test_log):
                with open(test_log) as f:
                    if sum(1 for l in f if l.strip()) >= n_comp:
                        continue
            
            print(f"  📝 Round {rdx}: Evaluating {n_comp} completions...")
            
            # Print one completion per task
            _seen_ns = set()
            with open(comp_file) as _cf:
                for _cl in _cf:
                    _cj = json.loads(_cl)
                    _ns = _cj['namespace']
                    if _ns not in _seen_ns:
                        _seen_ns.add(_ns)
                        _fn = _ns.split('.')[-1]
                        print(f"      ┌── {_fn} ({_ns})")
                        for _ln in _cj['completion'].split('\n'):
                            print(f"      │ {_ln}")
                        print(f"      └{'─'*50}")
            
            # DEBUG: Check filtered data
            if os.path.exists(filtered_data):
                with open(filtered_data) as _f:
                    _lines = _f.readlines()
                print(f"      [DEBUG] data_filtered.jsonl: {len(_lines)} entries")
                if _lines:
                    _first = json.loads(_lines[0])
                    print(f"      [DEBUG] First entry keys: {list(_first.keys())}")
                    print(f"      [DEBUG] First ns: {_first.get('namespace')}")
                    print(f"      [DEBUG] First completion_path: {_first.get('completion_path')}")
                    print(f"      [DEBUG] First tests: {_first.get('tests', [])[:2]}")
            else:
                print(f"      [DEBUG] ❌ data_filtered.jsonl NOT FOUND at {filtered_data}")
            
            # DEBUG: Check completion file
            with open(comp_file) as _f:
                _clines = _f.readlines()
            if _clines:
                _cfirst = json.loads(_clines[0])
                print(f"      [DEBUG] completion.jsonl: {len(_clines)} entries, first ns: {_cfirst.get('namespace')}")
            
            # pass@k
            r = subprocess.run([
                "python", f"{WORK_DIR}/traver/parser/pass_k.py",
                "--output_file", comp_file,
                "--log_file", test_log,
                "--data_file", filtered_data,
                "--source_code_root", Source_Code_Root,
                "--k", "1,3,5,10", "--n", str(POSTTEST_N),
            ], env=env, cwd=WORK_DIR, capture_output=True, text=True, timeout=600)
            print(f"      [DEBUG] pass_k exit code: {r.returncode}")
            
            # Summarize errors from pass_k output
            from collections import Counter
            _errors = Counter()
            for _line in r.stdout.split('\n'):
                if 'Execution Error' in _line:
                    _errors['Execution Error (test failed)'] += 1
                elif 'SyntaxError' in _line:
                    _errors['SyntaxError (bad code)'] += 1
                elif 'Timeout' in _line:
                    _errors['Timeout'] += 1
                elif 'Out of Memory' in _line:
                    _errors['OOM'] += 1
                elif 'Other Error' in _line:
                    _errors['Other Error'] += 1
                elif 'Skipping' in _line:
                    _errors['Skipped (missing file)'] += 1
                elif 'Pass@' in _line:
                    print(f"      {_line.strip()}")
            if _errors:
                print(f"      [ERRORS] {dict(_errors)}")
            
            # Show unique error details (deduplicated)
            _seen_errs = set()
            for _line in r.stdout.split('\n'):
                if 'PYTEST STDOUT:' in _line or 'PYTEST STDERR:' in _line:
                    _err_key = _line.strip()[:100]
                    if _err_key not in _seen_errs:
                        _seen_errs.add(_err_key)
                        print(f"      {_line.strip()[:200]}")
            
            # DEBUG: Check test_results after
            if os.path.exists(test_log):
                with open(test_log) as _f:
                    _rlines = _f.readlines()
                print(f"      [DEBUG] test_results.jsonl: {len(_rlines)} entries")
                if _rlines:
                    _rfirst = json.loads(_rlines[0])
                    print(f"      [DEBUG] First result: {_rfirst.get('Result')}, ns: {_rfirst.get('namespace')}")
            else:
                print(f"      [DEBUG] ❌ test_results.jsonl NOT CREATED")
            
            # recall@k
            dep_dir = f"{Dependency_Root}/{folder}"
            if os.path.exists(dep_dir):
                dep_log = f"{log_dir}/dependency_results.jsonl"
                subprocess.run([
                    "python", f"{WORK_DIR}/traver/parser/recall_k.py",
                    "--output_file", comp_file,
                    "--log_file", dep_log,
                    "--data_file", filtered_data,
                    "--source_code_root", Source_Code_Root,
                    "--dependency_data_root", Dependency_Root,
                    "--k", "1,3,5,10",
                ], env=env, cwd=WORK_DIR, capture_output=True, text=True, timeout=600)
        
        # ══════════════════════════════════════════════════
        # RESULTS SUMMARY
        # ══════════════════════════════════════════════════
        print(f"\n  {'─'*55}")
        print(f"  📊 {ns_prefix} / {level} Results:")
        for rdx in range(1, MAX_INTERACTION_ROUND + 1):
            test_log = f"{out_dir}/round_{rdx}/test_results.jsonl"
            comp_file = f"{out_dir}/round_{rdx}/completion.jsonl"
            if not os.path.exists(test_log) or not os.path.exists(comp_file):
                continue
            passed = defaultdict(set)
            with open(test_log) as f:
                for line in f:
                    js = json.loads(line)
                    if js.get('Result') == 'Pass':
                        passed[js['namespace']].add(js['completion'])
            results = {}
            with open(comp_file) as f:
                for line in f:
                    js = json.loads(line)
                    ns = js['namespace']
                    if ns not in results: results[ns] = 0
                    if ns in passed and js['completion'] in passed[ns]:
                        results[ns] += 1
            if not results: continue
            nonzero = sum(1 for v in results.values() if v > 0)
            metrics = []
            for k in [1, 3, 5, 10]:
                if k > POSTTEST_N: continue
                pak = np.mean([compute_pass_at_k(POSTTEST_N, c, k) for c in results.values()])
                metrics.append(f"P@{k}={pak*100:.1f}%")
            print(f"     round_{rdx}: {' | '.join(metrics)}  ({nonzero}/{len(results)} passed)")
    
    elapsed = time.time() - start_time
    print(f"\n{'='*65}")
    print(f"  ✅ {ns_prefix} → {folder} complete ({elapsed/60:.1f} min)")
    print(f"{'='*65}")


print("✅ run_full_pipeline() defined")
print(f"   Codegen: {CODEGEN_MODEL} via HF API")
print(f"   N={POSTTEST_N}, T={POSTTEST_TEMP}")
print(f"   Source code: {Source_Code_Root}")

---
### 🔍 searcharray (6 tasks → searcharray)
Pure Python/numpy project. All tests run on CPU.

In [ ]:
run_full_pipeline("searcharray")

### 🔍 Inspect searcharray completions across rounds

In [ ]:
# One completion per task per round — see how code improves with more tutoring
import json, os

ns_prefix = "searcharray"  # change to "xinhua", "gfpgan_model", etc.
level = "low_level"        # change to "med_level", "high_level"
base = f"{WORK_DIR}/output/student_posttest/{ns_prefix}/traver/{model_name}/{level}"

for rdx in range(1, 9):
    comp_file = f"{base}/round_{rdx}/completion.jsonl"
    if not os.path.exists(comp_file):
        break
    
    seen = set()
    print(f"\n{'='*70}")
    print(f"ROUND {rdx}")
    print(f"{'='*70}")
    
    with open(comp_file) as f:
        for line in f:
            js = json.loads(line)
            ns = js['namespace']
            if ns not in seen:
                seen.add(ns)
                short_ns = ns.split('.')[-1]
                print(f"\n── {short_ns} ({ns}) ──")
                print(js['completion'])
                print(f"{'─'*50}")


---
### 🔍 xinhua → UHGEval (1 task)
JSON statistics function. Tests run on CPU.

In [ ]:
run_full_pipeline("xinhua")

---
### 🔍 gfpgan_model → stable-diffusion-webui-forge (2 tasks)

In [ ]:
run_full_pipeline("gfpgan_model")

---
### 🔍 codeformer_model → stable-diffusion-webui-forge (1 task)

In [ ]:
run_full_pipeline("codeformer_model")

---
## 6. Phase 2: Run McMiner on Focused 4 Projects

Extract student code from the 4 project dialogues (from Drive), then use McMiner (Gemini 2.5 Flash) to identify misconceptions.

**Projects:** easyvolcap, searcharray, xinhua, gfpgan_model, codeformer_model


In [ ]:
import json, glob, re, os

PROJECTS = ["easyvolcap", "searcharray", "xinhua", "gfpgan_model", "codeformer_model"]
LEVELS = ["low_level", "med_level", "high_level"]

def extract_student_code_by_project(projects):
    """Extract code snippets from student messages in focused project dialogues."""
    samples = []
    for proj in projects:
        for level in LEVELS:
            f_path = f"{DRIVE_DIR}/output/dialogue_by_project/{proj}/traver/Llama-3.1-70B-Instruct/{level}/simulated_dialogs.jsonl"
            if not os.path.exists(f_path):
                print(f"  ⚠️  Not found: {proj}/{level}")
                continue
            count = 0
            with open(f_path) as f:
                for line in f:
                    data = json.loads(line.strip())
                    ns = data.get("namespace", "")
                    for turn_idx, turn in enumerate(data.get("conversation", [])):
                        if "student" not in turn:
                            continue
                        msg = turn["student"]
                        blocks = re.findall(r"```(?:python)?\s*(.*?)```", msg, re.DOTALL)
                        if not blocks and any(k in msg for k in ["def ", "class ", "import ", "return "]):
                            blocks = [msg]
                        for ci, code_str in enumerate(blocks):
                            code_str = code_str.strip()
                            if len(code_str) > 20:
                                samples.append({
                                    "namespace": ns, "project": proj, "level": level,
                                    "turn_index": turn_idx, "code_index": ci,
                                    "student_code": code_str, "source_file": f_path
                                })
                                count += 1
            if count:
                print(f"  ✅ {proj}/{level}: {count} code samples")
    return samples

print("📖 Extracting student code from 4 focused projects...")
code_samples = extract_student_code_by_project(PROJECTS)
print(f"\n📊 Total: {len(code_samples)} code samples")

# Per-project breakdown
from collections import Counter
proj_counts = Counter(s["project"] for s in code_samples)
for p, c in sorted(proj_counts.items()):
    print(f"  {p}: {c} samples")

# Save extracted code
out_dir = f"{DRIVE_DIR}/output/mcminer_focused"
os.makedirs(out_dir, exist_ok=True)
with open(f"{out_dir}/extracted_code.json", "w") as f:
    json.dump(code_samples, f, indent=2)
print(f"\n💾 Saved to {out_dir}/extracted_code.json")


In [ ]:
import time, re
import google.generativeai as genai

genai.configure(api_key=GOOGLE_API_KEY)

MCMINER_PROMPT = """You are an expert programming instructor. Analyze this student code for
programming MISCONCEPTIONS (fundamental misunderstandings, NOT just bugs/typos).

Student's code:
```python
{student_code}
```

If you find a misconception, respond:
<misconception>
<description>Concise description of the misconception</description>
<explanation>What the student believes vs reality</explanation>
<confidence>high/medium/low</confidence>
</misconception>

If no misconception (code is correct or just has typos): <misconception>NONE</misconception>"""

model = genai.GenerativeModel("gemini-2.5-flash")
results = []
total = len(code_samples)
print(f"🚀 McMiner on {total} samples from {len(PROJECTS)} projects...")

for i, s in enumerate(code_samples):
    t0 = time.time()
    prompt = MCMINER_PROMPT.format(student_code=s["student_code"])
    try:
        resp = model.generate_content(prompt)
        raw = resp.text
        elapsed = time.time() - t0
        detected = "NONE" not in raw
        print(f"  [{i+1}/{total}] ✓ {elapsed:.1f}s — {s['project']}/{s['level']} — {'DETECTED' if detected else 'none'}")
    except Exception as e:
        elapsed = time.time() - t0
        print(f"  [{i+1}/{total}] ⚠️ {elapsed:.1f}s — {e}")
        raw = "<misconception>NONE</misconception>"
        time.sleep(2)

    desc = re.search(r"<description>(.*?)</description>", raw, re.DOTALL)
    expl = re.search(r"<explanation>(.*?)</explanation>", raw, re.DOTALL)
    conf = re.search(r"<confidence>(.*?)</confidence>", raw, re.DOTALL)
    is_none = "NONE" in raw and not desc

    results.append({
        **s,
        "misconception_detected": not is_none,
        "misconception_description": desc.group(1).strip() if desc else None,
        "misconception_explanation": expl.group(1).strip() if expl else None,
        "confidence": conf.group(1).strip() if conf else None,
        "raw_response": raw
    })

# Save results
out_path = f"{out_dir}/misconception_results.json"
with open(out_path, "w") as f:
    json.dump(results, f, indent=2)

# Summary
detected = sum(1 for r in results if r["misconception_detected"])
print(f"\n{'='*60}")
print(f"📊 McMiner Results: {detected}/{len(results)} misconceptions ({100*detected/max(len(results),1):.1f}%)")
print(f"{'='*60}")

for proj in PROJECTS:
    proj_r = [r for r in results if r["project"] == proj]
    if not proj_r:
        continue
    d = sum(1 for r in proj_r if r["misconception_detected"])
    print(f"\n  {proj}: {d}/{len(proj_r)} misconceptions ({100*d/len(proj_r):.0f}%)")
    for level in LEVELS:
        lvl_r = [r for r in proj_r if r["level"] == level]
        if lvl_r:
            ld = sum(1 for r in lvl_r if r["misconception_detected"])
            print(f"    {level}: {ld}/{len(lvl_r)} ({100*ld/len(lvl_r):.0f}%)")

# Show sample misconceptions
print(f"\n{'='*60}")
print("🔍 Sample Misconceptions:")
print(f"{'='*60}")
shown = 0
for r in results:
    if r["misconception_detected"] and r.get("misconception_description") and shown < 5:
        print(f"\n  [{r['project']}/{r['level']}] {r['namespace']}")
        print(f"  Description: {r['misconception_description']}")
        print(f"  Confidence: {r.get('confidence', 'N/A')}")
        shown += 1

print(f"\n💾 Full results saved to {out_path}")


---
## 7. Phase 3: Inject Misconceptions & Re-run TRAVER

Augment the tutor prompt with misconception information from McMiner.

In [ ]:
# Build misconception lookup: namespace -> misconceptions
# Load from Drive if mcminer_results was lost (e.g., Colab disconnection)
import json

mcminer_save_path = f"{DRIVE_DIR}/output/mcminer/misconception_results.json"
if 'mcminer_results' not in dir() or not mcminer_results:
    print("📥 Loading McMiner results from Drive...")
    with open(mcminer_save_path) as f:
        mcminer_results = json.load(f)
    print(f"  ✓ Loaded {len(mcminer_results)} results")

misconception_lookup = {}
for r in mcminer_results:
    if r["misconception_detected"] and r["misconception_description"]:
        ns = r["namespace"]
        if ns not in misconception_lookup:
            misconception_lookup[ns] = []
        misconception_lookup[ns].append({
            "description": r["misconception_description"],
            "explanation": r["misconception_explanation"],
            "confidence": r["confidence"],
            "turn_index": r["turn_index"]})

lookup_path = f"{DRIVE_DIR}/output/mcminer/misconception_lookup.json"
with open(lookup_path, "w") as f:
    json.dump(misconception_lookup, f, indent=2)

print(f"📊 Lookup: {len(misconception_lookup)} namespaces, "
      f"{sum(len(v) for v in misconception_lookup.values())} misconceptions")


In [ ]:
# Create misconception-aware TRAVER wrapper script
script_content = f'''#!/usr/bin/env python3
"""Wrapper: injects McMiner misconceptions into tutor prompt."""
import json, sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), "traver"))
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

with open("{lookup_path}", "r") as f:
    MISC_LOOKUP = json.load(f)

from traver.utils.make_prompt import prompt_tutor as orig_prompt_tutor
import traver.utils.make_prompt as mp

def mc_prompt_tutor(d, tokenizer, setting="base", max_code_context=1024):
    prompt = orig_prompt_tutor(d, tokenizer, setting=setting, max_code_context=max_code_context)
    ns = d.get("namespace", "")
    if ns in MISC_LOOKUP and setting == "base":
        txt = "\\n\\n--- STUDENT MISCONCEPTION ALERT ---\\n"
        txt += "Detected misconceptions in this student\'s code:\\n"
        for i, m in enumerate(MISC_LOOKUP[ns], 1):
            txt += f"  {{i}}. {{m[\'description\']}}\\n"
            if m.get("explanation"):
                txt += f"     Context: {{m[\'explanation\']}}\\n"
        txt += ("\\nGuide the student to discover and correct these misconceptions "
               "step-by-step using Socratic questioning. Do NOT give direct answers.\\n"
               "--- END ALERT ---")
        prompt += txt
    return prompt

mp.prompt_tutor = mc_prompt_tutor
from traver.run_traver import parse_args, main
args = parse_args()
print(f"\\n🧠 McMiner-TRAVER: {{len(MISC_LOOKUP)}} namespaces with misconceptions")
main(args)
'''

script_path = f"{WORK_DIR}/run_traver_mcminer.py"
with open(script_path, "w") as f:
    f.write(script_content)
print(f"✅ Created: {script_path}")

In [ ]:
import subprocess, os, sys, pathlib, base64

# --- Patch model_utils.py with our fixed version ---
# Loads Mistral-7B in fp16 (no 4-bit quantization) to avoid
# bitsandbytes/.to() incompatibility on Colab
fixed_code = base64.b64decode("aW1wb3J0IG9zCmltcG9ydCB0b3JjaApmcm9tIHR5cGluZyBpbXBvcnQgTGlzdCwgT3B0aW9uYWwKZnJvbSB0cmFuc2Zvcm1lcnMgaW1wb3J0ICgKICAgIEF1dG9Nb2RlbEZvckNhdXNhbExNLCAKICAgIEF1dG9Ub2tlbml6ZXIsIAogICAgQml0c0FuZEJ5dGVzQ29uZmlnLAogICAgVHJhaW5lciwKKQpmcm9tIHBlZnQgaW1wb3J0ICgKICAgIExvcmFDb25maWcsCiAgICBwcmVwYXJlX21vZGVsX2Zvcl9rYml0X3RyYWluaW5nLAopCmZyb20gLm1vZGVsIGltcG9ydCBWZXJpZmllcgoKCmNsYXNzIFZlcmlmaWVyVHJhaW5lcihUcmFpbmVyKToKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBtb2RlbCwgYXJncywgdG9rZW5pemVyLCB0cmFpbl9kYXRhc2V0LCBldmFsX2RhdGFzZXQpOgogICAgICAgIHN1cGVyKCkuX19pbml0X18obW9kZWwsIGFyZ3MsCiAgICAgICAgICAgICAgICAgICAgICAgICB0b2tlbml6ZXI9dG9rZW5pemVyLAogICAgICAgICAgICAgICAgICAgICAgICAgdHJhaW5fZGF0YXNldD10cmFpbl9kYXRhc2V0LAogICAgICAgICAgICAgICAgICAgICAgICAgZXZhbF9kYXRhc2V0PWV2YWxfZGF0YXNldCkKCiAgICBkZWYgc2F2ZV9tb2RlbChzZWxmLCBvdXRwdXRfZGlyOiBPcHRpb25hbFtzdHJdID0gTm9uZSwgX2ludGVybmFsX2NhbGw6IGJvb2wgPSBGYWxzZSk6CiAgICAgICAgaWYgb3V0cHV0X2RpciBpcyBOb25lOgogICAgICAgICAgICBvdXRwdXRfZGlyID0gc2VsZi5hcmdzLm91dHB1dF9kaXIKICAgICAgICBvcy5tYWtlZGlycyhvdXRwdXRfZGlyLCBleGlzdF9vaz1UcnVlKQoKICAgICAgICBtb2RlbF90b19zYXZlID0gc2VsZi5tb2RlbAoKICAgICAgICBvdXRwdXRfbW9kZWxfZmlsZSA9IG9zLnBhdGguam9pbihvdXRwdXRfZGlyLCAicHl0b3JjaF9tb2RlbC5iaW4iKQogICAgICAgIHRvcmNoLnNhdmUobW9kZWxfdG9fc2F2ZS5zdGF0ZV9kaWN0KCksIG91dHB1dF9tb2RlbF9maWxlKQoKCmRlZiBsb2FkX21vZGVsKAogICAgYmFzZV9tb2RlbF9uYW1lX29yX3BhdGg6IHN0ciwKICAgIHRyYWluZWRfdmVyaWZpZXJfbW9kZWxfcGF0aDogc3RyID0gTm9uZSwKICAgIGxvcmFfcjogaW50ID0gOCwKICAgIGxvcmFfYWxwaGE6IGludCA9IDE2LAogICAgbG9yYV9kcm9wb3V0OiBmbG9hdCA9IDAuMDUsCiAgICBsb3JhX3RhcmdldF9tb2R1bGVzOiBMaXN0W3N0cl0gPSAgWyJxX3Byb2oiLCAidl9wcm9qIl0sCiAgICBmcDE2OiBib29sID0gVHJ1ZSwKICAgIGJmMTY6IGJvb2wgPSBGYWxzZSwKICAgIGdyYWRpZW50X2NoZWNrcG9pbnRpbmc6IGJvb2wgPSBGYWxzZQopOgogICAgIyBMb2FkIHRoZSBwcmUtdHJhaW5lZCBtb2RlbCBhbmQgdG9rZW5pemVyCiAgICBkZXZpY2VfbWFwID0gImF1dG8iCiAgICB3b3JsZF9zaXplID0gaW50KG9zLmVudmlyb24uZ2V0KCJXT1JMRF9TSVpFIiwgMSkpCiAgICBkZHAgPSB3b3JsZF9zaXplICE9IDEKICAgIGlmIGRkcDoKICAgICAgICBkZXZpY2VfbWFwID0geyIiOiBpbnQob3MuZW52aXJvbi5nZXQoIkxPQ0FMX1JBTksiKSBvciAwKX0KICAgIAogICAgY29tcHV0ZV9kdHlwZSA9ICgKICAgICAgICB0b3JjaC5mbG9hdDE2CiAgICAgICAgaWYgZnAxNgogICAgICAgIGVsc2UgKHRvcmNoLmJmbG9hdDE2IGlmIGJmMTYgZWxzZSB0b3JjaC5mbG9hdDMyKQogICAgKSAgICAKCiAgICAjIDQtYml0IHF1YW50aXphdGlvbiBmb3IgQ29sYWIgVDQvVjEwMCAoMTUtMTZHQiBWUkFNKQogICAgYm5iX2NvbmZpZyA9IEJpdHNBbmRCeXRlc0NvbmZpZygKICAgICAgICBsb2FkX2luXzRiaXQ9VHJ1ZSwKICAgICAgICBibmJfNGJpdF9xdWFudF90eXBlPSJuZjQiLAogICAgICAgIGJuYl80Yml0X2NvbXB1dGVfZHR5cGU9Y29tcHV0ZV9kdHlwZSwKICAgICAgICBibmJfNGJpdF91c2VfZG91YmxlX3F1YW50PUZhbHNlLAogICAgKQoKICAgIGJhc2VfbW9kZWwgPSBBdXRvTW9kZWxGb3JDYXVzYWxMTS5mcm9tX3ByZXRyYWluZWQoCiAgICAgICAgYmFzZV9tb2RlbF9uYW1lX29yX3BhdGgsCiAgICAgICAgcXVhbnRpemF0aW9uX2NvbmZpZz1ibmJfY29uZmlnLAogICAgICAgIGRldmljZV9tYXA9ZGV2aWNlX21hcCwKICAgICAgICB0b3JjaF9kdHlwZT1jb21wdXRlX2R0eXBlLAogICAgICAgIHVzZV9jYWNoZT1GYWxzZSwKICAgICAgICB0cnVzdF9yZW1vdGVfY29kZT1UcnVlCiAgICApCiAgICBsb3JhX2NvbmZpZyA9IExvcmFDb25maWcoCiAgICAgICAgcj1sb3JhX3IsCiAgICAgICAgbG9yYV9hbHBoYT1sb3JhX2FscGhhLAogICAgICAgIHRhcmdldF9tb2R1bGVzPWxvcmFfdGFyZ2V0X21vZHVsZXMsCiAgICAgICAgbG9yYV9kcm9wb3V0PWxvcmFfZHJvcG91dCwKICAgICAgICBiaWFzPSJub25lIiwKICAgICAgICB0YXNrX3R5cGU9IkNBVVNBTF9MTSIsCiAgICApCgogICAgIyBPbmx5IHByZXBhcmUgZm9yIHRyYWluaW5nIChjYXN0cyB0byBmcDMyIGZvciBzdGFibGUgZ3JhZGllbnRzKS4KICAgICMgU2tpcCBkdXJpbmcgaW5mZXJlbmNlIHRvIHNhdmUgVlJBTSDigJQgbm8gYmFja3Byb3AgbmVlZGVkLgogICAgaWYgdHJhaW5lZF92ZXJpZmllcl9tb2RlbF9wYXRoIGlzIE5vbmU6CiAgICAgICAgYmFzZV9tb2RlbCA9IHByZXBhcmVfbW9kZWxfZm9yX2tiaXRfdHJhaW5pbmcoCiAgICAgICAgICAgIGJhc2VfbW9kZWwsIHVzZV9ncmFkaWVudF9jaGVja3BvaW50aW5nPWdyYWRpZW50X2NoZWNrcG9pbnRpbmcpCiAgICAKICAgIGlmIG5vdCBkZHAgYW5kIHRvcmNoLmN1ZGEuZGV2aWNlX2NvdW50KCkgPiAxOgogICAgICAgICMga2VlcHMgVHJhaW5lciBmcm9tIHRyeWluZyBpdHMgb3duIERhdGFQYXJhbGxlbGlzbSB3aGVuIG1vcmUgdGhhbiAxIGdwdSBpcyBhdmFpbGFibGUKICAgICAgICBiYXNlX21vZGVsLmlzX3BhcmFsbGVsaXphYmxlID0gVHJ1ZQogICAgICAgIGJhc2VfbW9kZWwubW9kZWxfcGFyYWxsZWwgPSBUcnVlCgogICAgIyBTZXQgdG9rZW5pemVyJ3MgcGFkZGluZyB0b2tlbiBhbmQgcGFkZGluZyBzaWRlCiAgICB0b2tlbml6ZXIgPSBBdXRvVG9rZW5pemVyLmZyb21fcHJldHJhaW5lZCgKICAgICAgICBiYXNlX21vZGVsX25hbWVfb3JfcGF0aCwKICAgICAgICB0cnVuY2F0aW9uX3NpZGU9J2xlZnQnLCAgIyBzZXQgdG8gJ2xlZnQnIHRvIHRydW5jYXRlIHRoZSBpbnB1dCBmcm9tIHRoZSBsZWZ0CiAgICAgICAgdHJ1c3RfcmVtb3RlX2NvZGU9VHJ1ZQogICAgKQogICAgaWYgYmFzZV9tb2RlbC5jb25maWcubW9kZWxfdHlwZSA9PSAibGxhbWEiIG9yIGJhc2VfbW9kZWwuY29uZmlnLm1vZGVsX3R5cGUgPT0gIm1pc3RyYWwiOgogICAgICAgIHRva2VuaXplci5wYWRfdG9rZW4gPSB0b2tlbml6ZXIuZW9zX3Rva2VuCgogICAgIyBXcmFwIHRoZSBtb2RlbCB3aXRoIHRoZSBkZWZpbmVkIFBSTSBtb2RlbAogICAgdmVyaWZ5X21vZGVsID0gVmVyaWZpZXIoCiAgICAgICAgbW9kZWw9YmFzZV9tb2RlbCwKICAgICAgICBsb3JhX2NvbmZpZz1sb3JhX2NvbmZpZywKICAgICAgICB0b3JjaF9kdHlwZT1jb21wdXRlX2R0eXBlCiAgICApCgogICAgaWYgdHJhaW5lZF92ZXJpZmllcl9tb2RlbF9wYXRoIGlzIG5vdCBOb25lOgogICAgICAgIHByaW50KGYiTG9hZGluZyB0cmFpbmVkIHZlcmlmaWVyIG1vZGVsIGZyb20ge3RyYWluZWRfdmVyaWZpZXJfbW9kZWxfcGF0aH0iKQogICAgICAgIHN0YXRlX2RpY3QgPSB0b3JjaC5sb2FkKHRyYWluZWRfdmVyaWZpZXJfbW9kZWxfcGF0aCwgbWFwX2xvY2F0aW9uPSJjcHUiLCB3ZWlnaHRzX29ubHk9VHJ1ZSkKICAgICAgICB0cnk6CiAgICAgICAgICAgIHZlcmlmeV9tb2RlbC5sb2FkX3N0YXRlX2RpY3Qoc3RhdGVfZGljdCwgc3RyaWN0PUZhbHNlKQogICAgICAgICAgICBwcmludCgiTW9kZWwgbG9hZGVkIHN1Y2Nlc3NmdWxseSIpCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBlOgogICAgICAgICAgICBwcmludChmIldhcm5pbmc6IEVycm9yIGxvYWRpbmcgbW9kZWwgc3RhdGUgZGljdDoge2V9IikKICAgICAgICAgICAgbWlzc2luZ19rZXlzLCB1bmV4cGVjdGVkX2tleXMgPSB2ZXJpZnlfbW9kZWwubG9hZF9zdGF0ZV9kaWN0KHN0YXRlX2RpY3QsIHN0cmljdD1GYWxzZSkKICAgICAgICAgICAgaWYgbWlzc2luZ19rZXlzOgogICAgICAgICAgICAgICAgcHJpbnQoZiJNaXNzaW5nIGtleXM6IHttaXNzaW5nX2tleXN9IikKICAgICAgICAgICAgaWYgdW5leHBlY3RlZF9rZXlzOgogICAgICAgICAgICAgICAgcHJpbnQoZiJVbmV4cGVjdGVkIGtleXM6IHt1bmV4cGVjdGVkX2tleXN9IikKCiAgICAgICAgIyBNb3ZlIG9ubHkgdGhlIG5vbi1xdWFudGl6ZWQgdmVyaWZpZXIgcGFyYW1ldGVycyB0byBHUFUKICAgICAgICAjIChnYWluLCBiaWFzLCB2c2NvcmVfaGVhZCkuIFRoZSBxdWFudGl6ZWQgYmFzZV9tb2RlbCBpcyBhbHJlYWR5CiAgICAgICAgIyBvbiBHUFUgdmlhIGRldmljZV9tYXAsIGJ1dCBjdXN0b20gcGFyYW1zIGxvYWRlZCBmcm9tIENQVSBuZWVkIG1vdmluZy4KICAgICAgICBkZXZpY2UgPSBuZXh0KGJhc2VfbW9kZWwucGFyYW1ldGVycygpKS5kZXZpY2UKICAgICAgICB2ZXJpZnlfbW9kZWwuZ2FpbiA9IHRvcmNoLm5uLlBhcmFtZXRlcih2ZXJpZnlfbW9kZWwuZ2Fpbi50byhkZXZpY2UpKQogICAgICAgIHZlcmlmeV9tb2RlbC5iaWFzID0gdG9yY2gubm4uUGFyYW1ldGVyKHZlcmlmeV9tb2RlbC5iaWFzLnRvKGRldmljZSkpCiAgICAgICAgdmVyaWZ5X21vZGVsLnZzY29yZV9oZWFkID0gdmVyaWZ5X21vZGVsLnZzY29yZV9oZWFkLnRvKGRldmljZSkKICAgICAgICB2ZXJpZnlfbW9kZWwuZHJvcG91dCA9IHZlcmlmeV9tb2RlbC5kcm9wb3V0LnRvKGRldmljZSkKICAgICAgICB2ZXJpZnlfbW9kZWwuZXZhbCgpCgogICAgcmV0dXJuIHZlcmlmeV9tb2RlbCwgdG9rZW5pemVyCg==").decode()
mu_path = pathlib.Path(f"{WORK_DIR}/traver/verifier/model_utils.py")
mu_path.write_text(fixed_code)
print("✅ Patched model_utils.py: fp16 loading")

# Remove shadowing verifier dir if it exists
shadow_dir = pathlib.Path(f"{WORK_DIR}/verifier")
if shadow_dir.exists() and not (shadow_dir / "data_utils.py").exists():
    import shutil
    shutil.rmtree(shadow_dir)
    print("✅ Removed shadowing verifier/ directory")

os.makedirs(f"{WORK_DIR}/traver/utils", exist_ok=True)
with open(f"{WORK_DIR}/traver/__init__.py", "w") as f: pass
with open(f"{WORK_DIR}/traver/utils/__init__.py", "w") as f:
    f.write("from .utils import *\nfrom .make_prompt import *\n")

for level in STUDENT_LEVELS:
    print(f"\n{'='*60}")
    print(f"🧠 Phase 3: McMiner-TRAVER - {level}")
    print(f"{'='*60}")
    sys.stdout.flush()

    env = os.environ.copy()
    env["PYTHONPATH"] = f"{WORK_DIR}/traver:{WORK_DIR}"
    env["PYTHONUNBUFFERED"] = "1"
    env["HF_TOKEN"] = HF_TOKEN

    cmd = [
        "python", f"{WORK_DIR}/run_traver_mcminer.py",
        "--tutor_setting", "traver",
        "--namespace_file", f"{WORK_DIR}/prompt/namespaces.json",
        "--prompt_element_file", f"{WORK_DIR}/prompt/prompt_elements_final.jsonl",
        "--output_dir", f"{WORK_DIR}/output/dialogue_mcminer",
        "--verifier_base_model_path", MODEL_DIR + "/Mistral-7B-v0.1",
        "--verifier_model_dir", MODEL_DIR + "/Verifier-7B",
        "--tutor_model_name_or_path", TUTOR_MODEL_ID,
        "--tutor_num_responses", str(TUTOR_NUM_RESPONSES),
        "--student_model_name_or_path", STUDENT_MODEL_ID,
        "--student_setting", level,
        "--vllm_api_key", HF_TOKEN,
        "--vllm_endpoint_tutor", HF_API_BASE,
        "--vllm_endpoint_student", "local",
        "--show_description", "false",
        "--show_message", "true",
    ]

    process = subprocess.Popen(cmd, env=env, cwd=WORK_DIR,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    
    for line in iter(process.stdout.readline, ''):
        print(line, end='')
        sys.stdout.flush()
        
    process.stdout.close()
    returncode = process.wait()

    if returncode != 0:
        print(f"⚠️ {level} FAILED (exit code {returncode})")
    sys.stdout.flush()

print("\n✅ Phase 3 complete!")


---
## 8. Phase 4: Evaluate & Compare

In [ ]:
def load_dialogues(d):
    all_d = []
    for fp in glob.glob(f"{d}/**/simulated_dialogs.jsonl", recursive=True):
        with open(fp) as f:
            for line in f:
                data = json.loads(line.strip())
                for p in fp.split("/"):
                    if p in ["low_level", "med_level", "high_level"]:
                        data["student_level"] = p
                        break
                all_d.append(data)
    return all_d

def show_stats(dialogues, label):
    levels = {}
    for d in dialogues:
        lv = d.get("student_level", "unknown")
        if lv not in levels:
            levels[lv] = {"count": 0, "turns": 0}
        levels[lv]["count"] += 1
        levels[lv]["turns"] += len(d.get("conversation", []))
    total = sum(l["turns"] for l in levels.values())
    n = max(len(dialogues), 1)
    print(f"\n📊 {label}:")
    print(f"  Total: {len(dialogues)} dialogues, {total/n:.1f} avg turns")
    for lv, ld in sorted(levels.items()):
        print(f"  {lv}: {ld['count']} dialogues, {ld['turns']/max(ld['count'],1):.1f} avg turns")

print("="*60)
print("📈 EVALUATION COMPARISON")
print("="*60)
baseline = load_dialogues(f"{DRIVE_DIR}/output/dialogue/traver")
mcminer = load_dialogues(f"{DRIVE_DIR}/output/dialogue_mcminer/traver")
show_stats(baseline, "Baseline TRAVER")
show_stats(mcminer, "McMiner-TRAVER")

In [ ]:
# Full evaluation instructions
print("⚠️  Full pass@k evaluation requires the EvoCodeBench execution environment.")
print("   The dialogue statistics above provide a quick comparison.")
print()
print("For full evaluation, run these scripts:")
print(f"  1. cd {WORK_DIR}")
print(f"  2. bash scripts/run/run_pretest.sh")
print(f"  3. bash scripts/run/run_code_gen.sh")
print(f"  4. bash scripts/run/run_coding_test.sh")
print(f"  5. python scripts/eval/eval_TOR.py")

---
## 📋 Summary

| Phase | Description | Output |
|-------|-------------|--------|
| **1** | Baseline TRAVER dialogues | `output/dialogue/traver/` |
| **2** | McMiner misconception analysis | `output/mcminer/` |
| **3** | McMiner-aware TRAVER dialogues | `output/dialogue_mcminer/traver/` |
| **4** | Evaluation comparison | Stats above |

All outputs saved to Google Drive: `/content/drive/MyDrive/Coding-Tutor-Colab/output/`

**Session-safe**: Progress is checkpointed. Re-run cells if disconnected.